Cell 1: High-Speed Storage Extraction

In [ ]:
import os
import shutil
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Define your centralized Drive path and the fast local path
drive_zip_path = "/content/drive/MyDrive/BCI_dataset.zip"
local_zip_path = "/content/BCI_dataset.zip"

# The target directory (update this based on what the specific model requires)
local_extract_path = "/content/datasets_raw/BCI"

# 1. Transfer the zip to local high-speed storage
print("Pulling dataset from central storage...")
shutil.copy(drive_zip_path, local_zip_path)

# 2. Extract silently and overwrite
print("Extracting to local environment...")
os.makedirs(local_extract_path, exist_ok=True)
os.system(f"unzip -q -o {local_zip_path} -d {local_extract_path}")

print("Dataset is staged and ready for high-speed training.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pulling dataset from central storage...
Extracting to local environment...
Dataset is staged and ready for high-speed training.


Cell 2: Stratified Split Generation

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

def generate_stratified_splits(base_dir="/content/datasets_raw/BCI/BCI_dataset", output_dir="/content/datasets_raw/BCI/splits"):
    """
    Parses the BCI dataset to extract file paths and HER2 levels,
    creating static, stratified CSV splits for rigid model evaluation.
    """
    he_train_dir = os.path.join(base_dir, "HE", "train")
    ihc_train_dir = os.path.join(base_dir, "IHC", "train")
    he_test_dir = os.path.join(base_dir, "HE", "test")
    ihc_test_dir = os.path.join(base_dir, "IHC", "test")

    def get_file_data(he_dir, ihc_dir):
        data = []
        if not os.path.exists(he_dir):
            return data
        for filename in os.listdir(he_dir):
            if filename.endswith(".png"):
                he_path = os.path.join(he_dir, filename)
                ihc_path = os.path.join(ihc_dir, filename)

                # Extract HER2 expression level (e.g., '0', '1+', '2+', '3+')
                label = filename.split('_')[-1].replace('.png', '')

                data.append({
                    'he_path': he_path,
                    'ihc_path': ihc_path,
                    'her2_level': label
                })
        return data

    print("Parsing dataset files and extracting HER2 labels...")

    # 1. Process the existing static test set
    test_data = get_file_data(he_test_dir, ihc_test_dir)
    test_df = pd.DataFrame(test_data)

    # 2. Process the training set and stratify to create a validation split
    train_full_data = get_file_data(he_train_dir, ihc_train_dir)
    train_full_df = pd.DataFrame(train_full_data)

    # Stratified split: 85% Train, 15% Validation (locks the random seed)
    train_df, val_df = train_test_split(
        train_full_df,
        test_size=0.15,
        random_state=42,
        stratify=train_full_df['her2_level']
    )

    # 3. Export to CSV manifests
    os.makedirs(output_dir, exist_ok=True)
    train_df.to_csv(os.path.join(output_dir, "train_split.csv"), index=False)
    val_df.to_csv(os.path.join(output_dir, "val_split.csv"), index=False)
    test_df.to_csv(os.path.join(output_dir, "test_split.csv"), index=False)

    print(f"Static splits successfully generated in: {output_dir}")
    print(f"Distribution -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Execute the generation
generate_stratified_splits()

Parsing dataset files and extracting HER2 labels...
Static splits successfully generated in: /content/datasets_raw/BCI/splits
Distribution -> Train: 3311 | Val: 585 | Test: 977


Cell 3: Universal Standardization (512x512)

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

print("Standardizing the universal dataset to 512x512 based on global splits...")
splits_dir = "/content/datasets_raw/BCI/splits"
universal_data_dir = "/content/datasets_standardized/BCI_512"
os.makedirs(universal_data_dir, exist_ok=True)

for split in ['train', 'val', 'test']:
    split_csv = os.path.join(splits_dir, f"{split}_split.csv")
    if not os.path.exists(split_csv): continue

    df = pd.read_csv(split_csv)
    dir_A, dir_B = os.path.join(universal_data_dir, split, 'A'), os.path.join(universal_data_dir, split, 'B')
    os.makedirs(dir_A, exist_ok=True); os.makedirs(dir_B, exist_ok=True)

    for idx, row in df.iterrows():
        # Enforcing 512x512 resolution uniformly
        img_A = Image.open(row['he_path']).convert('RGB').resize((512, 512), Image.BICUBIC)
        img_B = Image.open(row['ihc_path']).convert('RGB').resize((512, 512), Image.BICUBIC)

        img_A.save(os.path.join(dir_A, f"{idx:05d}_A.png"))
        img_B.save(os.path.join(dir_B, f"{idx:05d}_B.png"))

print("Universal 512x512 dataset generated. All models must map to:", universal_data_dir)

Standardizing the universal dataset to 512x512 based on global splits...
Universal 512x512 dataset generated. All models must map to: /content/datasets_standardized/BCI_512


Cell 4: Kill Processes to clear GPU RAM (run when necessary)

In [ ]:
import subprocess, os, torch

# Kill any lingering python/training processes holding GPU memory
subprocess.run("kill -9 $(lsof /dev/nvidia0 2>/dev/null | awk 'NR>1{print $2}' | sort -u)",
               shell=True, capture_output=True)

# Clear PyTorch's own cache
torch.cuda.empty_cache()

import gc; gc.collect()

# Confirm free memory
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=memory.free,memory.total',
                        '--format=csv,noheader'], capture_output=True, text=True)
print("GPU memory status:", result.stdout.strip())

Cell 5: Dataset Restore (run at start of each new session)

In [1]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

drive_path = "/content/drive/MyDrive/BCI_Project_Final"

!rm -rf /content/datasets_standardized
os.makedirs("/content/datasets_standardized", exist_ok=True)

print("Extracting...")
!tar -xzf {drive_path}/BCI_512_Standardized.tar.gz -C /content/datasets_standardized

print("\nValidation:")
!echo "Train A:" $(ls /content/datasets_standardized/BCI_512/train/A | wc -l) pairs
!echo "Train B:" $(ls /content/datasets_standardized/BCI_512/train/B | wc -l) pairs
!echo "Val A  :" $(ls /content/datasets_standardized/BCI_512/val/A   | wc -l) pairs
!echo "Val B  :" $(ls /content/datasets_standardized/BCI_512/val/B   | wc -l) pairs
!echo "Test A :" $(ls /content/datasets_standardized/BCI_512/test/A  | wc -l) pairs
!echo "Test B :" $(ls /content/datasets_standardized/BCI_512/test/B  | wc -l) pairs

Mounted at /content/drive
Extracting...

Validation:
Train A: 3311 pairs
Train B: 3311 pairs
Val A  : 585 pairs
Val B  : 585 pairs
Test A : 977 pairs
Test B : 977 pairs


Cell 6: Clone BBDM Repo

In [10]:
import os

# Clone BBDM repo if not already present
if not os.path.exists('/content/BBDM'):
    !git clone https://github.com/xuekt98/BBDM.git /content/BBDM
    print("BBDM cloned.")
else:
    print("BBDM already present.")

# Install any missing dependencies
!pip install -q omegaconf einops

Cloning into '/content/BBDM'...
remote: Enumerating objects: 327, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 327 (delta 92), reused 63 (delta 63), pack-reused 185 (from 1)
Receiving objects: 100% (327/327), 1.09 MiB | 1.13 MiB/s, done.
Resolving deltas: 100% (114/114), done.
BBDM cloned.


Cell 7: Brownian Bridge Diffusion Model (Baseline DM) | Must Clone the BBDM Repo first ^^

In [15]:
!pip install -q pytorch_lightning

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BBDM — Brownian Bridge Diffusion Model  [A100]                        ║
# ║  KEY FIX: background Drive backup thread saves checkpoints every 5min  ║
# ║  during training — survives runtime disconnects.                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

MODE = "a100"
assert MODE in ("colab", "a100")

import os, re, glob, yaml, subprocess, time, sys, shutil, threading
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# ── Clone if missing ───────────────────────────────────────────────────────
if not os.path.exists('/content/BBDM'):
    print("Cloning BBDM...")
    os.system("git clone https://github.com/xuekt98/BBDM.git /content/BBDM")
os.chdir('/content/BBDM')
if '/content/BBDM' not in sys.path:
    sys.path.insert(0, '/content/BBDM')

# ══════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════
if MODE == "colab":
    IMAGE_SIZE     = 256; BATCH_SIZE = 1;  MODEL_CHANNELS = 64
    CHANNEL_MULT   = [1,2,3,4]; ATTN_RES = [16,32]; ACCUM = 1
    SAVE_INTERVAL  = 1; SAMPLE_INTERVAL = 1; VAL_INTERVAL = 5
    WALL_SECS      = 4*3600
    RESULTS_DIR    = "/content/results/BBDM_COLAB"
    DRIVE_CKPT_DIR = "/content/drive/MyDrive/BBDM_checkpoints_colab"
else:
    IMAGE_SIZE     = 256; BATCH_SIZE = 32; MODEL_CHANNELS = 128
    CHANNEL_MULT   = [1,1,2,2,4,4]; ATTN_RES = [8,16,32]; ACCUM = 1
    SAVE_INTERVAL  = 10; SAMPLE_INTERVAL = 10; VAL_INTERVAL = 10
    WALL_SECS      = 2*3600
    RESULTS_DIR    = "/content/results/BBDM_A100"
    DRIVE_CKPT_DIR = "/content/drive/MyDrive/BBDM_checkpoints_a100"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
nw = 2 if MODE == "colab" else 4

print(f"Mode   : {MODE.upper()} | {IMAGE_SIZE}px batch={BATCH_SIZE} "
      f"channels={MODEL_CHANNELS} wall={WALL_SECS//3600}h")
print(f"Drive  : {DRIVE_CKPT_DIR}")

# ══════════════════════════════════════════════════════════════════════════
# BACKGROUND DRIVE BACKUP THREAD
# Runs every 5 minutes during training, copies any new .pt files to Drive.
# Survives runtime disconnects — checkpoints are safe even if Colab crashes.
# ══════════════════════════════════════════════════════════════════════════
_stop_backup   = threading.Event()
_backed_up     = set()

def _drive_backup_loop():
    while not _stop_backup.is_set():
        try:
            # Search all .pt files BBDM creates internally
            ckpts = glob.glob('/content/BBDM/results/**/*.pt', recursive=True)
            for ckpt in ckpts:
                if ckpt not in _backed_up:
                    dst = os.path.join(DRIVE_CKPT_DIR, os.path.basename(ckpt))
                    shutil.copy2(ckpt, dst)
                    _backed_up.add(ckpt)
                    print(f"\n  [Drive Backup ✓] {os.path.basename(ckpt)}")
        except Exception as e:
            print(f"\n  [Drive Backup ✗] {e}")
        _stop_backup.wait(timeout=300)   # check every 5 minutes

_backup_thread = threading.Thread(target=_drive_backup_loop, daemon=True)
_backup_thread.start()
print("  Background Drive backup thread started (every 5 min)")

# ══════════════════════════════════════════════════════════════════════════
# PATCHES
# ══════════════════════════════════════════════════════════════════════════
print("\nApplying patches...")

for fp in glob.glob('runners/**/*.py', recursive=True):
    with open(fp) as f: c = f.read()
    if 'verbose=' in c:
        c = (c.replace('verbose=True,','').replace('verbose=True','')
              .replace('verbose=False,','').replace('verbose=False',''))
        with open(fp,'w') as f: f.write(c)

bbdm_path = 'model/BrownianBridge/BrownianBridgeModel.py'
with open(bbdm_path) as f: src = f.read()
old = 'self.denoise_fn = UNetModel(**vars(model_params.UNetParams))'
new = ('unet_kwargs = {k: v for k, v in vars(model_params.UNetParams).items()\n'
       '                          if k != "condition_key"}\n'
       '        self.denoise_fn = UNetModel(**unet_kwargs)')
if old in src:
    with open(bbdm_path,'w') as f: f.write(src.replace(old, new))
    print("Patch 2 applied.")
else:
    print("Patch 2 skipped.")

openai_path = 'model/BrownianBridge/base/modules/diffusionmodules/openaimodel.py'
with open(openai_path) as f: src = f.read()
if 'hspop' in src:
    src = (src.replace('hspop = hs.pop()','hsp = hs.pop()')
              .replace('th.cat([h, hspop]','th.cat([h, hsp]'))
    with open(openai_path,'w') as f: f.write(src)
    print("Patch 3 applied.")
else:
    print("Patch 3 skipped.")

for fp in (glob.glob('runners/**/*.py', recursive=True) +
           glob.glob('datasets/**/*.py', recursive=True)):
    with open(fp) as f: c = f.read()
    if 'DataLoader' in c and 'num_workers' in c:
        p = re.sub(r'num_workers\s*=\s*\d+', f'num_workers={nw}', c)
        if p != c:
            with open(fp,'w') as f: f.write(p)
            print(f"Patch 4: num_workers={nw} in {fp}")

# ══════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════
os.makedirs('datasets', exist_ok=True)
if not os.path.exists('datasets/__init__.py'):
    open('datasets/__init__.py','w').close()

dataset_code = '''
import os, sys, random
sys.path.insert(0, '/content/BBDM')
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF

try:
    from Register import Registers
    _has_register = True
except Exception:
    _has_register = False

def _register(cls):
    if _has_register:
        for name in [cls.__name__, 'custom_aligned', 'CustomAligned',
                     'CustomSingleDataset', 'CustomAlignedDataset',
                     'CustomInpaintingDataset']:
            try:
                if hasattr(Registers.datasets, 'register'):
                    Registers.datasets.register(cls)
            except Exception: pass
            try:
                Registers.datasets[name] = cls
            except Exception: pass
    return cls

@_register
class CustomAlignedDataset(Dataset):
    def __init__(self, dataset_config, stage="train"):
        cfg          = dataset_config if isinstance(dataset_config, dict) else vars(dataset_config)
        self.path    = cfg.get("dataset_path", cfg.get("root", ""))
        self.size    = int(cfg.get("image_size", 256))
        self.norm    = cfg.get("to_normal", True)
        self.augment = cfg.get("flip", False) and stage == "train"
        split        = "train" if stage == "train" else stage
        dir_A        = os.path.join(self.path, split, "A")
        dir_B        = os.path.join(self.path, split, "B")
        self.pairs   = [
            (os.path.join(dir_A, f),
             os.path.join(dir_B, f.replace("_A.", "_B.")))
            for f in sorted(os.listdir(dir_A))
            if f.endswith(".png") and
            os.path.exists(os.path.join(dir_B, f.replace("_A.", "_B.")))
        ]
        print(f"  [BBDM dataset | {stage}] {len(self.pairs)} pairs")

    def __len__(self): return len(self.pairs)

    def _to_tensor(self, img):
        t = T.ToTensor()(img.resize((self.size, self.size), Image.BICUBIC))
        return t * 2 - 1 if self.norm else t

    def __getitem__(self, idx):
        p_he, p_ihc = self.pairs[idx]
        he  = Image.open(p_he).convert("RGB")
        ihc = Image.open(p_ihc).convert("RGB")
        if self.augment and random.random() > 0.5: he, ihc = TF.hflip(he), TF.hflip(ihc)
        if self.augment and random.random() > 0.5: he, ihc = TF.vflip(he), TF.vflip(ihc)
        fname = os.path.basename(p_ihc)
        return (self._to_tensor(ihc), fname), (self._to_tensor(he), fname)

CustomSingleDataset     = CustomAlignedDataset
CustomInpaintingDataset = CustomAlignedDataset
'''

import importlib
if os.path.exists('datasets/__pycache__'):
    shutil.rmtree('datasets/__pycache__')
for mod in list(sys.modules.keys()):
    if 'datasets' in mod: del sys.modules[mod]
with open('datasets/custom.py','w') as f: f.write(dataset_code)
print("\ndatasets/custom.py written.")

# Smoke test
print("\n─── Dataset smoke test ──────────────────────────────────────────")
try:
    from datasets.custom import CustomAlignedDataset
    _cfg = {"dataset_path":"/content/datasets_standardized/BCI_512",
            "image_size":IMAGE_SIZE, "to_normal":True, "flip":False}
    _ds  = CustomAlignedDataset(_cfg, stage="train")
    (ihc_t, ihc_n),(he_t, he_n) = _ds[0]
    print(f"  IHC : {ihc_t.shape}  H&E : {he_t.shape}")
    print("  ✓ PASSED")
except Exception as e:
    print(f"  ✗ FAILED: {e}"); raise

# Registers check
print("\n─── Registers check ─────────────────────────────────────────────")
try:
    from Register import Registers
    for n in ['CustomAlignedDataset','CustomSingleDataset','CustomInpaintingDataset']:
        try: print(f"  ✓ {n}"); _ = Registers.datasets[n]
        except: print(f"  ✗ {n} NOT found")
except Exception as e:
    print(f"  {e}")

# ══════════════════════════════════════════════════════════════════════════
# YAML CONFIG
# ══════════════════════════════════════════════════════════════════════════
dataset_config = {
    'dataset_name':   'BCI_512',
    'dataset_type':   'CustomAlignedDataset',
    'dataset_config': {
        'dataset_path': '/content/datasets_standardized/BCI_512',
        'image_size':   IMAGE_SIZE, 'channels': 3,
        'to_normal':    True, 'flip': True,
    }
}
config = {
    'runner': 'BBDMRunner',
    'training': {
        'use_DDP': False, 'device': ['cuda:0'],
        'n_epochs': 50, 'n_steps': 9999999,
        'save_interval':       SAVE_INTERVAL,
        'sample_interval':     SAMPLE_INTERVAL,
        'validation_interval': VAL_INTERVAL,
        'accumulate_grad_batches': ACCUM,
    },
    'args':    {'sample_at_start': False, 'save_top': False},
    'testing': {'clip_denoised': True, 'sample_num': 1},
    'data': {
        **dataset_config,
        'train': {'batch_size': BATCH_SIZE, 'shuffle': True,  'num_workers': nw},
        'val':   {'batch_size': 1,          'shuffle': False, 'num_workers': nw},
        'test':  {'batch_size': 1,          'shuffle': False, 'num_workers': nw},
    },
    'dataset': dataset_config,
    'model': {
        'model_name': f'BBDM_{MODE.upper()}', 'model_type': 'BBDM',
        'normalize_latent': False, 'only_load_latent_mean_std': False,
        'BB': {
            'optimizer': {'optimizer':'Adam','lr':1e-4,'weight_decay':0,
                          'amsgrad':False,'beta1':0.9,'beta2':0.999,'eps':1e-8},
            'lr_scheduler': {'factor':0.5,'patience':3000,'threshold':1e-4,'min_lr':1e-6},
            'params': {
                'mt_type':'linear','objective':'grad','loss_type':'l1',
                'skip_sample':True,'sample_type':'linear','sample_step':200,
                'num_timesteps':1000,'eta':0.0,'max_var':1.0,
                'UNetParams': {
                    'image_size':IMAGE_SIZE,'in_channels':6,
                    'model_channels':MODEL_CHANNELS,'out_channels':3,
                    'num_res_blocks':2,'attention_resolutions':ATTN_RES,
                    'channel_mult':CHANNEL_MULT,'conv_resample':True,
                    'dims':2,'num_heads':8,'use_checkpoint':True,
                    'use_new_attention_order':False,
                    'condition_key':'SpatialRescaler',
                }
            }
        }
    },
    'train': {'n_epochs':5000,'batch_size':BATCH_SIZE,'lr':1e-4,'save_interval':SAVE_INTERVAL}
}
os.makedirs('configs', exist_ok=True)
with open('configs/My_Method_config.yaml','w') as f:
    yaml.dump(config, f, default_flow_style=False)
print(f"\nConfig written → {IMAGE_SIZE}px | batch={BATCH_SIZE} | wall={WALL_SECS//3600}h")

# ══════════════════════════════════════════════════════════════════════════
# LAUNCH
# ══════════════════════════════════════════════════════════════════════════
CRITICAL_KEYWORDS = ['Epoch:','loss:','saved','checkpoint','Checkpoint',
                     'Error','Exception','Traceback','start training','Total Number']
PROGRESS_SAMPLE_RATE = 200

def is_critical(line): return any(k in line for k in CRITICAL_KEYWORDS)
def is_progress(line): return '%|' in line or ('it/s' in line and '%' in line)

print(f"\nLaunching BBDM [{MODE.upper()}] — {WALL_SECS//3600}h wall cap...\n")
wall_start = time.time()

process = subprocess.Popen(
    ["timeout", str(WALL_SECS),
     "python3", "-u", "main.py",
     "--config", "configs/My_Method_config.yaml",
     "--train", "--gpu_ids", "0"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
pc = 0
for line in process.stdout:
    s = line.rstrip()
    if not s: continue
    if is_progress(s):
        pc += 1
        if pc % PROGRESS_SAMPLE_RATE == 0: print(s)
    elif is_critical(s):
        print(s)
process.wait()

# Stop backup thread + do one final full backup pass
_stop_backup.set()
print(f"\nDone. Elapsed: {(time.time()-wall_start)/60:.1f} min")
print("\n─── Final Drive backup ───────────────────────────────────────────")
all_ckpts = glob.glob('/content/BBDM/results/**/*.pt', recursive=True)
for ckpt in all_ckpts:
    dst = os.path.join(DRIVE_CKPT_DIR, os.path.basename(ckpt))
    shutil.copy2(ckpt, dst)
    print(f"  Saved → {os.path.basename(ckpt)}")
print(f"  Total: {len(all_ckpts)} checkpoints on Drive at {DRIVE_CKPT_DIR}")

# ══════════════════════════════════════════════════════════════════════════
# POST-TRAINING EVAL — compute PSNR + SSIM from sample_to_eval images
# FIX: images in sample_to_eval are INDIVIDUAL (not side-by-side grids)
# FIX: search all sample_to_eval dirs and use the one with most images
# ══════════════════════════════════════════════════════════════════════════
print("\n─── Post-training evaluation ─────────────────────────────────────")
import torch, math, numpy as np
from pathlib import Path
import torch.nn.functional as F
from PIL import Image as PILImage
import torchvision.transforms as T

def _psnr(a,b):
    return 10*math.log10(4.0/(((a.clamp(-1,1)-b.clamp(-1,1))**2).mean().item()+1e-8))
def _ssim(a,b):
    k,p=11,5; ma,mb=F.avg_pool2d(a,k,1,p),F.avg_pool2d(b,k,1,p)
    sa=F.avg_pool2d(a**2,k,1,p)-ma**2; sb=F.avg_pool2d(b**2,k,1,p)-mb**2
    sab=F.avg_pool2d(a*b,k,1,p)-ma*mb; C1,C2=0.01**2,0.03**2
    return (((2*ma*mb+C1)*(2*sab+C2))/((ma**2+mb**2+C1)*(sa+sb+C2)).clamp_min(1e-8)).mean().item()
to_t = lambda p: T.ToTensor()(PILImage.open(p).convert('RGB'))*2-1

# Find the sample_to_eval directory with the most images
all_eval_dirs = list(Path('/content/BBDM/results').rglob('sample_to_eval'))
print(f"  Found {len(all_eval_dirs)} sample_to_eval dirs")

best_dir   = None
best_count = 0
for d in all_eval_dirs:
    imgs = list(d.glob('*.png'))
    print(f"    {d}  →  {len(imgs)} images")
    if len(imgs) > best_count:
        best_count = len(imgs); best_dir = d

if best_dir and best_count > 0:
    recon_dir  = Path(RESULTS_DIR) / "eval" / "recon"
    target_dir = Path(RESULTS_DIR) / "eval" / "target"
    recon_dir.mkdir(parents=True, exist_ok=True)
    target_dir.mkdir(parents=True, exist_ok=True)

    gen_imgs  = sorted(best_dir.glob('*.png'))
    real_ihcs = sorted(Path("/content/datasets_standardized/BCI_512/val/B").glob("*.png"))

    psnr_s, ssim_s, saved = [], [], 0
    for i,(gf,rf) in enumerate(zip(gen_imgs[:100], real_ihcs)):
        img = PILImage.open(gf)
        # BBDM sample_to_eval images are individual — no cropping needed
        # But if width > height it might be a grid — detect and crop
        W,H = img.size
        if W > H:
            img = img.crop((W//2, 0, W, H))   # take right half if side-by-side
        img.save(recon_dir / f"{i:05d}.png")
        shutil.copy2(rf, target_dir / f"{i:05d}.png")
        p_t = to_t(recon_dir  / f"{i:05d}.png").unsqueeze(0)
        g_t = to_t(target_dir / f"{i:05d}.png").unsqueeze(0)
        psnr_s.append(_psnr(p_t,g_t)); ssim_s.append(_ssim(p_t,g_t))
        saved += 1

    print(f"\n  PSNR : {np.mean(psnr_s):.4f} dB")
    print(f"  SSIM : {np.mean(ssim_s):.6f}")
    print(f"  {saved} pairs → {recon_dir.parent}")
    print(f"  Run LPIPS+FID cell to complete all 4 metrics")

    # Backup eval images to Drive too
    eval_drive = Path(DRIVE_CKPT_DIR) / "eval_images"
    eval_drive.mkdir(exist_ok=True)
    (eval_drive/"recon").mkdir(exist_ok=True)
    (eval_drive/"target").mkdir(exist_ok=True)
    for f in recon_dir.glob("*.png"):
        shutil.copy2(f, eval_drive/"recon"/f.name)
    for f in target_dir.glob("*.png"):
        shutil.copy2(f, eval_drive/"target"/f.name)
    print(f"  Eval images backed up → {eval_drive}")
else:
    print("  ✗ No generated images found in sample_to_eval")
    print("    Possible fix: check /content/BBDM/results/ manually")
    print(f"    All results dirs: {list(Path('/content/BBDM/results').iterdir())}")

print(f"\nResults : {RESULTS_DIR}")
print(f"Drive   : {DRIVE_CKPT_DIR}")

Mode   : A100 | 256px batch=32 channels=128 wall=2h
Drive  : /content/drive/MyDrive/BBDM_checkpoints_a100
  Background Drive backup thread started (every 5 min)

Applying patches...
Patch 2 skipped.
Patch 3 skipped.

datasets/custom.py written.

─── Dataset smoke test ──────────────────────────────────────────
  [BBDM dataset | train] 3311 pairs
  IHC : torch.Size([3, 256, 256])  H&E : torch.Size([3, 256, 256])
  ✓ PASSED

─── Registers check ─────────────────────────────────────────────
  ✓ CustomAlignedDataset
  ✓ CustomSingleDataset
  ✓ CustomInpaintingDataset

Config written → 256px | batch=32 | wall=2h

Launching BBDM [A100] — 2h wall cap...

Total Number of parameter: 120.25M
start training BBDM_A100 on BCI_512, 103 iters per epoch
 76%|███████▌  | 445/585 [00:19<00:05, 23.35it/s]
saving latest checkpoint...
 52%|█████▏    | 307/585 [00:13<00:12, 22.90it/s]
saving latest checkpoint...
 28%|██▊       | 166/585 [00:07<00:18, 22.55it/s]
saving latest checkpoint...
  4%|▍         | 2

Cell 8: PST-Diff + Spatial Transformer Network (STN)

In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PST-Diff + STN  [TRIPLE MODE: colab / newton / a100]       ║
# ║  Same 2h wall cap, 50-epoch max, CKPT every 10 epochs       ║
# ║  as HistDiT, ScoreTopo, BBDM — fully comparable             ║
# ╚══════════════════════════════════════════════════════════════╝

MODE = "a100"   # "colab" → T4 debug  |  "newton" → HPC  |  "a100" → Colab A100

import gc, torch
for _var in ['he_enc','unet','stn','opt','sched','scaler','train_loader','val_loader']:
    if _var in globals(): del globals()[_var]
torch.cuda.empty_cache(); gc.collect()
_mem = torch.cuda.memory_allocated()/1024**3
print(f"GPU memory before build: {_mem:.2f} GiB  (should be near 0)")

assert MODE in ("colab","newton","a100")

import os, csv, time, math, random, shutil
import numpy as np
from datetime import datetime
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.checkpoint import checkpoint as grad_checkpoint
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")

torch.cuda.empty_cache(); gc.collect()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
SEED = 42; random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════
if MODE == "colab":
    IMAGE_SIZE      = 64
    NGF             = 32
    BATCH_SIZE      = 1;    ACCUM = 4
    T_DIFF          = 100;  N_EPOCHS = 3
    LR              = 1e-4; BETA_START = 1e-4; BETA_END = 0.02
    LAMBDA_CFG      = 0.15; LAMBDA_STN_REG = 0.5
    SAVE_EVERY      = 50;   SAMPLE_EVERY = 200
    MAX_VAL_BATCHES = 8;    VAL_DDIM_STEPS = 10
    USE_AMP         = True; USE_GRAD_CKPT = True; USE_STN = False
    MAX_WALL_SECS   = 4*3600 - 300
    CKPT_EVERY_N    = 1;    EXPLICIT_CKPT_EPOCHS = {8}
    RESULTS_DIR     = "/content/results/PST_STN_Diff_COLAB"
    DRIVE_CKPT      = "/content/drive/MyDrive/PST_STN_checkpoints_colab"

elif MODE == "a100":
    IMAGE_SIZE      = 256   # matches ScoreTopo / BBDM Colab runs
    NGF             = 128
    BATCH_SIZE      = 24;    ACCUM = 4    # eff batch = 8
    T_DIFF          = 1000; N_EPOCHS = 50
    LR              = 1e-4; BETA_START = 1e-4; BETA_END = 0.02
    LAMBDA_CFG      = 0.15; LAMBDA_STN_REG = 0.5
    SAVE_EVERY      = 100;  SAMPLE_EVERY = 500
    MAX_VAL_BATCHES = 16;   VAL_DDIM_STEPS = 50
    USE_AMP         = True; USE_GRAD_CKPT = False  # keep True at 256px even on A100
    USE_STN         = True
    MAX_WALL_SECS   = 2*3600               # 2h — matches other Colab models
    CKPT_EVERY_N    = 10;   EXPLICIT_CKPT_EPOCHS = {8}
    RESULTS_DIR     = "/content/results/PST_STN_Diff_A100"
    DRIVE_CKPT      = "/content/drive/MyDrive/PST_STN_checkpoints_a100"

else:  # newton
    IMAGE_SIZE      = 512
    NGF             = 128
    BATCH_SIZE      = 4;    ACCUM = 1
    T_DIFF          = 1000; N_EPOCHS = 100
    LR              = 1e-4; BETA_START = 1e-4; BETA_END = 0.02
    LAMBDA_CFG      = 0.15; LAMBDA_STN_REG = 0.5
    SAVE_EVERY      = 100;  SAMPLE_EVERY = 500
    MAX_VAL_BATCHES = 64;   VAL_DDIM_STEPS = 50
    USE_AMP         = True; USE_GRAD_CKPT = False; USE_STN = True
    MAX_WALL_SECS   = 999999
    CKPT_EVERY_N    = 1;    EXPLICIT_CKPT_EPOCHS = {8}
    RESULTS_DIR     = "/scratch/ja396977/results/PST_STN_Diff"
    DRIVE_CKPT      = "/scratch/ja396977/checkpoints/PST_STN"

AMP_DTYPE = (torch.bfloat16 if MODE in ("newton","a100") and
             torch.cuda.is_bf16_supported() else torch.float16)
DATA_ROOT  = "/content/datasets_standardized/BCI_512"
CKPT_DIR   = os.path.join(RESULTS_DIR, "checkpoints")
SAMPLE_DIR = os.path.join(RESULTS_DIR, "samples")
LOG_CSV    = os.path.join(RESULTS_DIR, "metrics_iter.csv")
EPOCH_CSV  = os.path.join(RESULTS_DIR, "metrics_epoch.csv")
for d in [CKPT_DIR, SAMPLE_DIR, DRIVE_CKPT]: os.makedirs(d, exist_ok=True)

print(f"Device : {DEVICE} | GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Mode   : {MODE.upper()} | {IMAGE_SIZE}px NGF={NGF} T={T_DIFF} "
      f"AMP={USE_AMP} GradCkpt={USE_GRAD_CKPT} STN={USE_STN}")

# ════════════════════════════════════════════════════════════════
# 1. DATASET  — fixed: self.pairs, plain (he, ihc) tensors
# ════════════════════════════════════════════════════════════════
class BCIFolderDataset(Dataset):
    def __init__(self, split, image_size=IMAGE_SIZE, augment=True,
                 explicit_files=None, force_dir=None):
        base     = force_dir or os.path.join(DATA_ROOT, split)
        dir_A    = os.path.join(base, "A")
        dir_B    = os.path.join(base, "B")
        all_files = explicit_files or sorted(
            f for f in os.listdir(dir_A) if f.endswith(".png"))
        # Only keep pairs where both A and B exist
        self.pairs = [
            (os.path.join(dir_A, f),
             os.path.join(dir_B, f.replace("_A.", "_B.")))
            for f in all_files
            if os.path.exists(os.path.join(dir_B, f.replace("_A.", "_B.")))
        ]
        self.augment = augment; self.size = image_size
        print(f"  [{split:5s}] {len(self.pairs)} pairs")

    def __len__(self): return len(self.pairs)

    def _to_tensor(self, img):
        return T.ToTensor()(img.resize((self.size, self.size), Image.BICUBIC)) * 2 - 1

    def __getitem__(self, idx):
        p_he, p_ihc = self.pairs[idx]
        he  = Image.open(p_he).convert("RGB")
        ihc = Image.open(p_ihc).convert("RGB")
        if self.augment and random.random() > 0.5: he, ihc = TF.hflip(he), TF.hflip(ihc)
        if self.augment and random.random() > 0.5: he, ihc = TF.vflip(he), TF.vflip(ihc)
        return self._to_tensor(he), self._to_tensor(ihc)  # (H&E, IHC) plain tensors

print("\nLoading datasets...")
_val_dir_A = os.path.join(DATA_ROOT, "val", "A")
if os.path.exists(_val_dir_A):
    train_ds = BCIFolderDataset("train", augment=True)
    val_ds   = BCIFolderDataset("val",   augment=False)
else:
    print("  [INFO] No val/ dir — carving last 15% of train as val")
    _all = sorted(f for f in os.listdir(os.path.join(DATA_ROOT,"train","A")) if f.endswith(".png"))
    _nv  = max(50, int(len(_all)*0.15)); _td = os.path.join(DATA_ROOT,"train")
    train_ds = BCIFolderDataset("train", augment=True,
                                explicit_files=_all[:-_nv], force_dir=_td)
    val_ds   = BCIFolderDataset("val",   augment=False,
                                explicit_files=_all[-_nv:], force_dir=_td)

_pin     = (MODE in ("newton","a100"))
_persist = (MODE in ("newton","a100"))
_nw      = 2 if MODE == "colab" else 4
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=_nw, pin_memory=_pin, drop_last=True,
                          persistent_workers=_persist)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=_nw, pin_memory=_pin,
                          persistent_workers=_persist)

iters_per_epoch = len(train_loader)
total_opt_steps = max(1, (N_EPOCHS * iters_per_epoch) // max(ACCUM, 1))
print(f"Train iters/epoch : {iters_per_epoch} | Total opt steps : {total_opt_steps}")

# ════════════════════════════════════════════════════════════════
# 2. DIFFUSION SCHEDULE
# ════════════════════════════════════════════════════════════════
class DiffusionSchedule:
    def __init__(self, T=T_DIFF, b0=BETA_START, b1=BETA_END, device=DEVICE):
        betas = torch.linspace(b0, b1, T, device=device)
        alphas = 1.0 - betas; ab = torch.cumprod(alphas, dim=0)
        ab_prev = F.pad(ab[:-1], (1,0), value=1.0)
        self.T = T; self.betas = betas; self.alpha_bar = ab
        self.sqrt_ab = ab.sqrt(); self.sqrt_1mab = (1-ab).sqrt()
        self.post_var = betas * (1-ab_prev) / (1-ab).clamp(1e-8)

    def q_sample(self, x0, t, noise=None):
        if noise is None: noise = torch.randn_like(x0)
        return (self.sqrt_ab[t].view(-1,1,1,1)*x0 +
                self.sqrt_1mab[t].view(-1,1,1,1)*noise, noise)

    @torch.no_grad()
    def ddim_sample(self, model_fn, cond, shape, ddim_steps=50):
        x = torch.randn(shape, device=DEVICE)
        seq = list(range(self.T-1, -1, -(self.T//ddim_steps)))
        for i, ti in enumerate(seq):
            tb  = torch.full((shape[0],), ti, device=DEVICE, dtype=torch.long)
            eps = model_fn(x, tb, cond); ab = self.alpha_bar[ti]
            abp = (self.alpha_bar[seq[i+1]] if i+1 < len(seq)
                   else torch.tensor(1.0, device=DEVICE))
            x0p = ((x-(1-ab).sqrt()*eps)/ab.sqrt().clamp(1e-8)).clamp(-1,1)
            x   = abp.sqrt()*x0p + (1-abp).sqrt()*eps
        return x

schedule = DiffusionSchedule()

# ════════════════════════════════════════════════════════════════
# 3. ARCHITECTURE
#    Key fixes vs original:
#    - _flash_attn: adds .contiguous() → prevents "stride-1" crash on A100
#    - AAM.forward: asymmetric Q/K (no upsampling ctx) → prevents 65K-token OOM
# ════════════════════════════════════════════════════════════════
def _flash_attn(q, k, v):
    """Force fp16 + contiguous before SDPA — required for Flash Attention on A100."""
    orig = q.dtype
    q = q.half().contiguous()
    k = k.half().contiguous()
    v = v.half().contiguous()
    return F.scaled_dot_product_attention(q, k, v).to(orig)

class TimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim,dim*4), nn.SiLU(), nn.Linear(dim*4,dim*4))
    def forward(self, t):
        h = self.dim//2
        freq = torch.exp(-math.log(10000)*torch.arange(h,device=t.device)/max(h-1,1))
        emb  = t.float()[:,None]*freq[None]
        return self.mlp(torch.cat([emb.sin(), emb.cos()], dim=-1))

class ResBlock(nn.Module):
    def __init__(self, ic, oc, td):
        super().__init__()
        self.n1 = nn.GroupNorm(min(8,ic),ic); self.c1 = nn.Conv2d(ic,oc,3,1,1)
        self.tp = nn.Linear(td, oc*2)
        self.n2 = nn.GroupNorm(min(8,oc),oc); self.c2 = nn.Conv2d(oc,oc,3,1,1)
        self.sk = nn.Conv2d(ic,oc,1) if ic!=oc else nn.Identity()
        nn.init.zeros_(self.c2.weight); nn.init.zeros_(self.c2.bias)
    def forward(self, x, te):
        h = self.c1(F.silu(self.n1(x))); t = self.tp(F.silu(te))[:,:,None,None]
        sc,sh = t.chunk(2,dim=1); h = self.n2(h)*(1+sc)+sh
        return self.c2(F.silu(h)) + self.sk(x)

class SelfAttn(nn.Module):
    def __init__(self, ch, heads=4):
        super().__init__(); self.heads = heads
        self.norm = nn.GroupNorm(min(8,ch),ch); self.qkv = nn.Conv2d(ch,ch*3,1)
        self.proj = nn.Conv2d(ch,ch,1)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)
    def forward(self, x):
        B,C,H,W = x.shape; q,k,v = self.qkv(self.norm(x)).chunk(3,dim=1)
        d = C//self.heads
        def re(t): return t.view(B,self.heads,d,H*W).transpose(2,3)
        out = _flash_attn(re(q),re(k),re(v)).transpose(2,3).reshape(B,C,H,W)
        return x + self.proj(out)

class AAM(nn.Module):
    """
    Asymmetric Attention Module.
    FIX: Do NOT upsample cond to match x resolution — use asymmetric Q/K.
    This prevents the 65K-token O(N²) attention explosion at full resolution.
    Q has H×W tokens (x resolution), K/V keep cond's native lower resolution.
    """
    def __init__(self, ch, cch, heads=4):
        super().__init__(); self.heads = heads
        self.nq   = nn.GroupNorm(min(8,ch),ch); self.nkv = nn.GroupNorm(min(8,cch),cch)
        self.q    = nn.Conv2d(ch,ch,1); self.k = nn.Conv2d(cch,ch,1); self.v = nn.Conv2d(cch,ch,1)
        self.proj = nn.Conv2d(ch,ch,1); self.gate = nn.Parameter(torch.zeros(1))
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)
    def forward(self, x, cond):
        B,C,H,W = x.shape
        Hc,Wc   = cond.shape[-2], cond.shape[-1]
        # Asymmetric: Q at (H,W), K/V at (Hc,Wc) — no upsampling ctx
        q = self.q(self.nq(x)); k = self.k(self.nkv(cond)); v = self.v(cond)
        d = C // self.heads
        def re_q(t):  return t.view(B,self.heads,d,H*W).transpose(2,3)
        def re_kv(t): return t.view(B,self.heads,d,Hc*Wc).transpose(2,3)
        out = _flash_attn(re_q(q), re_kv(k), re_kv(v))
        out = out.transpose(2,3).reshape(B,C,H,W)
        g   = torch.sigmoid(self.gate)
        return g*self.proj(out) + (1-g)*x

class LatentTransfer(nn.Module):
    def __init__(self, nc, hc):
        super().__init__()
        self.norm  = nn.GroupNorm(min(8,hc),hc)
        self.gamma = nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),
                                    nn.Linear(hc,nc*2),nn.SiLU(),nn.Linear(nc*2,nc))
        self.beta  = nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),
                                    nn.Linear(hc,nc*2),nn.SiLU(),nn.Linear(nc*2,nc))
        nn.init.ones_(self.gamma[-1].weight);  nn.init.zeros_(self.gamma[-1].bias)
        nn.init.zeros_(self.beta[-1].weight);  nn.init.zeros_(self.beta[-1].bias)
    def forward(self, hn, hh):
        if hh.shape[-2:] != hn.shape[-2:]:
            hh = F.interpolate(hh, hn.shape[-2:], mode="bilinear", align_corners=False)
        hhn = self.norm(hh)
        return self.gamma(hhn)[:,:,None,None]*hn + self.beta(hhn)[:,:,None,None]

class HEEncoder(nn.Module):
    def __init__(self, ngf=NGF):
        super().__init__()
        self.e1 = nn.Sequential(nn.Conv2d(3,ngf,3,1,1),nn.GroupNorm(min(8,ngf),ngf),nn.SiLU())
        self.e2 = nn.Sequential(nn.Conv2d(ngf,ngf*2,3,2,1),nn.GroupNorm(min(8,ngf*2),ngf*2),nn.SiLU())
        self.e3 = nn.Sequential(nn.Conv2d(ngf*2,ngf*4,3,2,1),nn.GroupNorm(min(8,ngf*4),ngf*4),nn.SiLU())
        self.e4 = nn.Sequential(nn.Conv2d(ngf*4,ngf*4,3,2,1),nn.GroupNorm(min(8,ngf*4),ngf*4),nn.SiLU())
    def forward(self, x):
        f1=self.e1(x); f2=self.e2(f1); f3=self.e3(f2); f4=self.e4(f3)
        return f1, f2, f3, f4

class PSTDiffUNet(nn.Module):
    def __init__(self, ngf=NGF, t_mult=4, use_ckpt=USE_GRAD_CKPT):
        super().__init__(); self.use_ckpt=use_ckpt; td=ngf*t_mult
        ch = [ngf, ngf*2, ngf*4, ngf*4]
        self.te=TimeEmb(ngf); self.ic=nn.Conv2d(3,ch[0],3,1,1)
        self.e1=nn.ModuleList([ResBlock(ch[0],ch[0],td),ResBlock(ch[0],ch[0],td)])
        self.d1=nn.Conv2d(ch[0],ch[0],3,2,1)
        self.e2=nn.ModuleList([ResBlock(ch[0],ch[1],td),ResBlock(ch[1],ch[1],td)])
        self.d2=nn.Conv2d(ch[1],ch[1],3,2,1)
        self.e3=nn.ModuleList([ResBlock(ch[1],ch[2],td),ResBlock(ch[2],ch[2],td),SelfAttn(ch[2])])
        self.d3=nn.Conv2d(ch[2],ch[2],3,2,1)
        self.e4=nn.ModuleList([ResBlock(ch[2],ch[3],td),ResBlock(ch[3],ch[3],td),SelfAttn(ch[3])])
        self.b1=ResBlock(ch[3],ch[3],td); self.ba=SelfAttn(ch[3])
        self.lt=LatentTransfer(ch[3],ch[3]); self.b2=ResBlock(ch[3],ch[3],td)
        self.u3=nn.ConvTranspose2d(ch[3],ch[2],2,2); self.a3=AAM(ch[2],ch[3])
        self.r3=nn.ModuleList([ResBlock(ch[2]*2,ch[2],td),ResBlock(ch[2],ch[2],td),SelfAttn(ch[2])])
        self.u2=nn.ConvTranspose2d(ch[2],ch[1],2,2); self.a2=AAM(ch[1],ch[3])
        self.r2=nn.ModuleList([ResBlock(ch[1]*2,ch[1],td),ResBlock(ch[1],ch[1],td)])
        self.u1=nn.ConvTranspose2d(ch[1],ch[0],2,2); self.a1=AAM(ch[0],ch[1])
        self.r1=nn.ModuleList([ResBlock(ch[0]*2,ch[0],td),ResBlock(ch[0],ch[0],td)])
        self.on=nn.GroupNorm(min(8,ch[0]),ch[0]); self.oc=nn.Conv2d(ch[0],3,3,1,1)
        nn.init.zeros_(self.oc.weight); nn.init.zeros_(self.oc.bias)
    def _run(self, x, blks, te):
        for b in blks:
            if isinstance(b, SelfAttn): x = b(x)
            elif self.use_ckpt and self.training: x = grad_checkpoint(b,x,te,use_reentrant=False)
            else: x = b(x,te)
        return x
    def forward(self, xt, t, hf):
        f1,f2,f3,f4 = hf; te = self.te(t)
        h  = self.ic(xt); h1 = self._run(h,  self.e1, te)
        h2 = self._run(self.d1(h1), self.e2, te)
        h3 = self._run(self.d2(h2), self.e3, te)
        h4 = self._run(self.d3(h3), self.e4, te)
        b  = self.b2(self.lt(self.ba(self.b1(h4,te)), f4), te)
        d  = self._run(torch.cat([self.a3(self.u3(b),f4), h3],1), self.r3, te)
        d  = self._run(torch.cat([self.a2(self.u2(d),f3), h2],1), self.r2, te)
        d  = self._run(torch.cat([self.a1(self.u1(d),f2), h1],1), self.r1, te)
        return self.oc(F.silu(self.on(d)))

# ════════════════════════════════════════════════════════════════
# 4. STN
# ════════════════════════════════════════════════════════════════
class STN(nn.Module):
    def __init__(self):
        super().__init__()
        self.loc = nn.Sequential(
            nn.Conv2d(6,32,7,2,3),nn.ReLU(inplace=True),
            nn.Conv2d(32,64,5,2,2),nn.ReLU(inplace=True),
            nn.Conv2d(64,128,3,2,1),nn.ReLU(inplace=True),nn.AdaptiveAvgPool2d(4))
        self.fc = nn.Sequential(nn.Flatten(),nn.Linear(128*16,256),nn.ReLU(inplace=True),nn.Linear(256,6))
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.copy_(torch.tensor([1.,0.,0.,0.,1.,0.]))
    def forward(self, rough, real):
        theta = self.fc(self.loc(torch.cat([rough.detach(),real],1))).view(-1,2,3)
        grid  = F.affine_grid(theta, real.size(), align_corners=False)
        return F.grid_sample(real, grid, align_corners=False, padding_mode="border"), theta

def stn_reg_loss(theta):
    ident = torch.tensor([[1.,0.,0.],[0.,1.,0.]], device=theta.device)
    return F.mse_loss(theta, ident.unsqueeze(0).expand_as(theta))

def cfg_loss(pred, target):
    def lm(x):
        return torch.log1p(torch.fft.fftshift(
            torch.fft.fft2(x,norm="ortho"),dim=(-2,-1)).abs())
    B,C,H,W = pred.shape
    yy = torch.arange(H,device=pred.device).float()-H//2
    xx = torch.arange(W,device=pred.device).float()-W//2
    mask = torch.exp(-(yy[:,None]**2+xx[None,:]**2)/(2*(H/6.0)**2)).unsqueeze(0).unsqueeze(0)
    return (mask*(lm(pred)-lm(target)).abs()).mean()

# ════════════════════════════════════════════════════════════════
# 5. MODEL ASSEMBLY
# ════════════════════════════════════════════════════════════════
print(f"\nBuilding PST-Diff + STN  [{MODE.upper()} mode]  STN={USE_STN}...")
he_enc = HEEncoder(ngf=NGF).to(DEVICE)
unet   = PSTDiffUNet(ngf=NGF, use_ckpt=USE_GRAD_CKPT).to(DEVICE)
stn    = STN().to(DEVICE) if USE_STN else None
def cp(m): return sum(p.numel() for p in m.parameters())/1e6
print(f"  HEEncoder   : {cp(he_enc):.2f}M")
print(f"  PSTDiffUNet : {cp(unet):.2f}M")
print(f"  STN         : {cp(stn):.2f}M" if USE_STN else "  STN         : disabled")
all_params = (list(he_enc.parameters()) + list(unet.parameters()) +
              (list(stn.parameters()) if USE_STN else []))
opt    = AdamW(all_params, lr=LR, betas=(0.9,0.999), weight_decay=1e-4)
sched  = CosineAnnealingLR(opt, T_max=total_opt_steps, eta_min=LR*0.01)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

# ════════════════════════════════════════════════════════════════
# 6. METRICS + CSV
# ════════════════════════════════════════════════════════════════
def psnr(a, b):
    return 10*math.log10(4.0/(((a.clamp(-1,1)-b.clamp(-1,1))**2).mean().item()+1e-8))

def ssim_fast(a, b):
    k,p = 11,5; ma,mb = F.avg_pool2d(a,k,1,p), F.avg_pool2d(b,k,1,p)
    sa  = F.avg_pool2d(a**2,k,1,p)-ma**2; sb = F.avg_pool2d(b**2,k,1,p)-mb**2
    sab = F.avg_pool2d(a*b, k,1,p)-ma*mb; C1,C2 = 0.01**2, 0.03**2
    return (((2*ma*mb+C1)*(2*sab+C2))/((ma**2+mb**2+C1)*(sa+sb+C2)).clamp_min(1e-8)).mean().item()

def init_csv(p, f):
    with open(p,"w",newline="",buffering=1) as fp: csv.writer(fp).writerow(f)
def append_csv(p, r):
    with open(p,"a",newline="",buffering=1) as fp: csv.writer(fp).writerow(r)

init_csv(LOG_CSV,   ["wall_time","epoch","iter","global_step",
                     "loss_total","loss_diff","loss_cfg","loss_stn"])
init_csv(EPOCH_CSV, ["wall_time","epoch","mean_total","mean_diff",
                     "mean_cfg","mean_stn","val_psnr","val_ssim"])

# ════════════════════════════════════════════════════════════════
# 7. HELPERS
# ════════════════════════════════════════════════════════════════
def save_samples(epoch, he, x0, aihc, rihc, step=0, tag="epoch"):
    def n(t): return ((t.detach().cpu().clamp(-1,1)+1)/2)
    vutils.save_image(
        vutils.make_grid(torch.cat([n(he),n(x0),n(aihc),n(rihc)]),
                         nrow=he.shape[0], padding=2),
        os.path.join(SAMPLE_DIR, f"{tag}_ep{epoch:04d}_s{step:07d}.png"))

@torch.no_grad()
def validate(epoch, mb=MAX_VAL_BATCHES, ds=VAL_DDIM_STEPS):
    he_enc.eval(); unet.eval(); ps,ss = [],[]
    for i,(he,ihc) in enumerate(val_loader):
        if i >= mb: break
        he,ihc = he.to(DEVICE), ihc.to(DEVICE)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            feats = he_enc(he)
            fake  = schedule.ddim_sample(
                lambda x,t,_: unet(x,t,feats), feats, ihc.shape, ddim_steps=ds)
        ps.append(psnr(fake,ihc)); ss.append(ssim_fast(fake,ihc))
        torch.cuda.empty_cache()
    he_enc.train(); unet.train()
    return float(np.mean(ps)), float(np.mean(ss))

def save_ckpt(path, epoch, opt_step, val_psnr, val_ssim):
    torch.save({"epoch":epoch,"opt_step":opt_step,"mode":MODE,
                "he_enc":he_enc.state_dict(),"unet":unet.state_dict(),
                "stn":stn.state_dict() if USE_STN else None,
                "opt":opt.state_dict(),"sched":sched.state_dict(),
                "scaler":scaler.state_dict(),
                "val_psnr":val_psnr,"val_ssim":val_ssim,
                "config":{"IMAGE_SIZE":IMAGE_SIZE,"NGF":NGF,"T_DIFF":T_DIFF,
                          "MODE":MODE,"USE_STN":USE_STN}}, path)
    print(f"  [Ckpt] → {path}")

def backup(ckpt):
    try:
        shutil.copy2(ckpt, os.path.join(DRIVE_CKPT, os.path.basename(ckpt)))
        print(f"  [Backup] saved")
    except Exception as e:
        print(f"  [Backup] failed: {e}")

# ════════════════════════════════════════════════════════════════
# 8. TRAINING LOOP
# ════════════════════════════════════════════════════════════════
print("\n"+"="*60)
print(f"  PST-Diff + STN  [{MODE.upper()}]  STN={'ON' if USE_STN else 'OFF'}")
print(f"  {IMAGE_SIZE}px  NGF={NGF}  T={T_DIFF}  Epochs={N_EPOCHS}  EffBatch={BATCH_SIZE*ACCUM}")
print(f"  Ckpt/val every {CKPT_EVERY_N} epochs | epoch 8 explicit | wall={MAX_WALL_SECS//3600}h")
print("="*60+"\n")

global_step  = 0
opt_step     = 0
wall_start   = time.time()
timeout_flag = False
last_val_psnr = 0.0
last_val_ssim = 0.0
last_epoch    = 0

for epoch in range(1, N_EPOCHS+1):
    if timeout_flag: break
    ep = {"total":[],"diff":[],"cfg":[],"stn":[]}
    opt.zero_grad()
    _vh,_vi = next(iter(val_loader)); _vh,_vi = _vh.to(DEVICE),_vi.to(DEVICE)

    for it, (real_he, real_ihc) in enumerate(train_loader):
        global_step += 1
        real_he  = real_he.to(DEVICE,  non_blocking=True)
        real_ihc = real_ihc.to(DEVICE, non_blocking=True)
        t = torch.randint(0, T_DIFF, (real_he.shape[0],), device=DEVICE)

        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            he_feats = he_enc(real_he)

        # STN branch
        if USE_STN:
            with torch.no_grad():
                _xtr,_ = schedule.q_sample(real_ihc, t)
                with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                    _er = unet(_xtr, t, tuple(f.detach() for f in he_feats))
                _ab  = schedule.alpha_bar[t].view(-1,1,1,1)
                x0r  = ((_xtr-(1-_ab).sqrt()*_er)/_ab.sqrt().clamp(1e-8)).clamp(-1,1)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                aligned_ihc,theta = stn(x0r, real_ihc)
                loss_stn = stn_reg_loss(theta) * LAMBDA_STN_REG
        else:
            aligned_ihc = real_ihc
            loss_stn    = torch.tensor(0.0, device=DEVICE)

        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            x_t, eps_true = schedule.q_sample(aligned_ihc.detach(), t)
            eps_pred  = unet(x_t, t, he_feats)
            loss_diff = F.l1_loss(eps_pred, eps_true)
            ab   = schedule.alpha_bar[t].view(-1,1,1,1)
            x0p  = ((x_t-(1-ab).sqrt()*eps_pred)/ab.sqrt().clamp(1e-8)).clamp(-1,1)
            loss_cfg = cfg_loss(x0p, aligned_ihc.detach()) * LAMBDA_CFG
            loss     = loss_diff + loss_cfg + loss_stn

        scaler.scale(loss / max(ACCUM,1)).backward()
        if global_step % max(ACCUM,1) == 0:
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(all_params, 1.0)
            scaler.step(opt); scaler.update()
            sched.step(); opt.zero_grad(); opt_step += 1

        for k,v in zip(["total","diff","cfg","stn"],[loss,loss_diff,loss_cfg,loss_stn]):
            ep[k].append(v.item())

        if global_step % SAVE_EVERY == 0:
            ts = datetime.now().isoformat(timespec="seconds")
            em = (time.time()-wall_start)/60
            append_csv(LOG_CSV, [ts,epoch,it+1,global_step] +
                       [f"{v.item():.6f}" for v in [loss,loss_diff,loss_cfg,loss_stn]])
            print(f"[{ts}] Ep{epoch}/{N_EPOCHS} It{it+1} {em:.1f}m | "
                  f"loss={loss.item():.4f} diff={loss_diff.item():.4f} "
                  f"cfg={loss_cfg.item():.4f}")

        if global_step % SAMPLE_EVERY == 0:
            save_samples(epoch, real_he, x0p, aligned_ihc, real_ihc,
                         step=global_step, tag="iter")

        if time.time()-wall_start > MAX_WALL_SECS:
            print(f"\n[TIMEOUT] Wall cap reached at epoch {epoch} iter {it+1}.")
            timeout_flag = True; break

    # ── Epoch end ────────────────────────────────────────────────────────
    last_epoch = epoch
    ml = {k: float(np.mean(v)) for k, v in ep.items()}
    torch.cuda.empty_cache(); gc.collect()

    if (epoch % CKPT_EVERY_N == 0 or
            epoch in EXPLICIT_CKPT_EPOCHS or
            timeout_flag):
        val_psnr, val_ssim = validate(epoch)
        last_val_psnr = val_psnr; last_val_ssim = val_ssim

        ts = datetime.now().isoformat(timespec="seconds")
        append_csv(EPOCH_CSV,
                   [ts, epoch] +
                   [f"{ml[k]:.6f}" for k in ["total","diff","cfg","stn"]] +
                   [f"{val_psnr:.4f}", f"{val_ssim:.6f}"])

        ckpt = os.path.join(CKPT_DIR, f"epoch_{epoch:04d}.pt")
        save_ckpt(ckpt, epoch, opt_step, val_psnr, val_ssim)

        if epoch in EXPLICIT_CKPT_EPOCHS:
            explicit = ckpt.replace(".pt", "_explicit.pt")
            shutil.copy2(ckpt, explicit)
            print(f"  [Ckpt] Explicit epoch {epoch} → {explicit}")

        with torch.no_grad():
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                _f = he_enc(_vh)
                _s = schedule.ddim_sample(
                    lambda x,t,_: unet(x,t,_f), _f, _vi.shape,
                    ddim_steps=VAL_DDIM_STEPS)
                _a,_ = stn(_s.clamp(-1,1),_vi) if USE_STN else (_vi,None)
        save_samples(epoch, _vh, _s, _a, _vi, step=global_step, tag="epoch")
        backup(ckpt)

    elapsed = (time.time()-wall_start)/60
    print(f"\n{'='*60}")
    print(f"  Epoch {epoch}/{N_EPOCHS} [{MODE.upper()}] | {elapsed:.1f}min")
    print(f"  Loss  total={ml['total']:.4f}  diff={ml['diff']:.4f}  "
          f"cfg={ml['cfg']:.4f}  stn={ml['stn']:.4f}")
    if epoch % CKPT_EVERY_N == 0 or epoch in EXPLICIT_CKPT_EPOCHS:
        print(f"  Val   PSNR={val_psnr:.4f} dB  SSIM={val_ssim:.6f}")
    print(f"{'='*60}\n")

    if timeout_flag: break

# ── Final timeout checkpoint ──────────────────────────────────────────────
if timeout_flag and last_epoch > 0:
    final = os.path.join(CKPT_DIR, f"final_epoch{last_epoch}.pt")
    save_ckpt(final, last_epoch, opt_step, last_val_psnr, last_val_ssim)
    backup(final)
    print(f"  [Ckpt] Final checkpoint → {final}")

print("\nTraining complete.")
print(f"  Mode     : {MODE.upper()}")
print(f"  Iter CSV : {LOG_CSV}")
print(f"  Epoch CSV: {EPOCH_CSV}")
print(f"  Ckpts    : {CKPT_DIR}")
print(f"  Samples  : {SAMPLE_DIR}")

GPU memory before build: 1.57 GiB  (should be near 0)
Device : cuda | GPU : NVIDIA A100-SXM4-80GB
Mode   : A100 | 256px NGF=128 T=1000 AMP=True GradCkpt=False STN=True

Loading datasets...
  [train] 3311 pairs
  [val  ] 585 pairs
Train iters/epoch : 137 | Total opt steps : 1712

Building PST-Diff + STN  [A100 mode]  STN=True...
  HEEncoder   : 3.84M
  PSTDiffUNet : 64.93M
  STN         : 0.66M

  PST-Diff + STN  [A100]  STN=ON
  256px  NGF=128  T=1000  Epochs=50  EffBatch=96
  Ckpt/val every 10 epochs | epoch 8 explicit | wall=2h

[2026-05-06T13:47:07] Ep1/50 It100 2.5m | loss=0.5934 diff=0.5827 cfg=0.0107

  Epoch 1/50 [A100] | 3.4min
  Loss  total=0.6626  diff=0.6526  cfg=0.0099  stn=0.0000

[2026-05-06T13:49:38] Ep2/50 It63 5.0m | loss=0.3455 diff=0.3375 cfg=0.0080

  Epoch 2/50 [A100] | 6.9min
  Loss  total=0.3314  diff=0.3232  cfg=0.0082  stn=0.0000

[2026-05-06T13:52:09] Ep3/50 It26 7.5m | loss=0.1387 diff=0.1311 cfg=0.0076
[2026-05-06T13:54:39] Ep3/50 It126 10.0m | loss=0.1254 d

Cell 9: Score-Based Diffusion + Topological Consistency

In [12]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Score-Based Diffusion + Topological Consistency              ║
# ║  [DUAL MODE: colab / newton]                                            ║
# ║                                                                          ║
# ║  Key differences from Cell 7 (PST-Diff):                               ║
# ║   1. Score function parameterization + SNR-weighted loss               ║
# ║      (emphasises lower-noise denoising steps vs uniform DDPM L1)       ║
# ║   2. Cross-attention H&E conditioning (no AAM / LT)                    ║
# ║   3. Topology Consistency Loss — soft Euler characteristic             ║
# ║      at N thresholds (replaces CFG frequency loss)                     ║
# ║   4. MI Contrastive Loss — InfoNCE between H&E encoder features        ║
# ║      and predicted IHC decoder features (replaces STN in colab mode)   ║
# ║                                                                          ║
# ║  Sources: MIU-Diff (arxiv 2506.23184), TA-GAN (arxiv 2601.02806)      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ▶▶  SET THIS BEFORE RUNNING  ◀◀
MODE = "a100"   # was "colab"
assert MODE in ("colab", "newton", "a100")  # add "a100"
# "colab"  → low-memory test on T4
                 # "newton" → full-resolution A100 training
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Cleanup: release any stale models / DataLoader workers from prior runs ──
import gc, torch
for _var in ['he_enc', 'score_unet', 'mi_proj_he', 'mi_proj_ihc',
             'opt', 'sched', 'scaler', 'train_loader', 'val_loader']:
    if _var in globals(): del globals()[_var]
torch.cuda.empty_cache(); gc.collect()
_mem = torch.cuda.memory_allocated() / 1024**3
print(f"GPU memory before build: {_mem:.2f} GiB  (should be near 0)")

# ── Imports ────────────────────────────────────────────────────────────────
import os, csv, time, math, random
import numpy as np
from datetime import datetime
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.checkpoint import checkpoint as grad_checkpoint

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")

# ── Environment ────────────────────────────────────────────────────────────
torch.cuda.empty_cache(); gc.collect()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════
if MODE == "colab":
    IMAGE_SIZE      = 64      # T4-safe
    NGF             = 128      # very narrow for testing
    BATCH_SIZE      = 1
    ACCUM           = 2       # eff batch = 2
    T_DIFF          = 100     # noise levels
    N_EPOCHS        = 3
    LR              = 1e-4
    # Score-based noise schedule (geometric, NCSN-style)
    SIGMA_MIN       = 0.01
    SIGMA_MAX       = 10.0
    # Loss weights
    LAMBDA_TOPO     = 0.10    # topology consistency weight
    LAMBDA_MI       = 0.05    # MI contrastive weight
    N_TOPO_THRESH   = 5       # number of topology threshold levels
    N_MI_PATCHES    = 32      # patches for MI InfoNCE
    # Validation
    SAVE_EVERY      = 50
    SAMPLE_EVERY    = 200
    MAX_VAL_BATCHES = 8
    VAL_DDIM_STEPS  = 10
    # Memory
    USE_AMP         = True
    USE_GRAD_CKPT   = True
    # Paths
    MAX_WALL_SECS   = 4*3600 - 300
    RESULTS_DIR     = "/content/results/ScoreTopo_Diff_COLAB"
    DRIVE_CKPT      = "/content/drive/MyDrive/ScoreTopo_checkpoints_colab"

elif MODE == "a100":
    IMAGE_SIZE      = 256
    # PATCH_SIZE      = 16     # N = (512/16)² = 1024 tokens
    NGF             = 128
    BATCH_SIZE      = 4
    ACCUM           = 2        # eff batch = 64
    T_DIFF          = 1000
    SIGMA_MIN       = 0.01
    SIGMA_MAX       = 50.0
    N_EPOCHS        = 50       # wall clock controls stop
    LR              = 1e-4
    LAMBDA_TOPO     = 0.10
    LAMBDA_MI       = 0.05
    N_TOPO_THRESH   = 10
    N_MI_PATCHES    = 64
    SAVE_EVERY      = 100
    SAMPLE_EVERY    = 500
    MAX_VAL_BATCHES = 16
    VAL_DDIM_STEPS  = 50
    USE_AMP         = True
    USE_GRAD_CKPT   = True    # A100 80GB — not needed
    MAX_WALL_SECS   = 2*3600
    EXPLICIT_CKPT_EPOCHS = {8}
    RESULTS_DIR     = "/content/results/ScoreTopo_Diff_A100"
    DRIVE_CKPT      = "/content/drive/MyDrive/ScoreTopo_checkpoints_a100"

AMP_DTYPE    = torch.bfloat16 if MODE == "a100" else (
               torch.bfloat16 if MODE == "newton" and torch.cuda.is_bf16_supported()
               else torch.float16)
CKPT_EVERY_N = 10
if MODE == "colab": EXPLICIT_CKPT_EPOCHS = {8}  # already set for newton/a100 above
DATA_ROOT  = "/content/datasets_standardized/BCI_512"
CKPT_DIR   = os.path.join(RESULTS_DIR, "checkpoints")
SAMPLE_DIR = os.path.join(RESULTS_DIR, "samples")
LOG_CSV    = os.path.join(RESULTS_DIR, "metrics_iter.csv")
EPOCH_CSV  = os.path.join(RESULTS_DIR, "metrics_epoch.csv")
for d in [CKPT_DIR, SAMPLE_DIR, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)

print(f"Device : {DEVICE} | GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Mode   : {MODE.upper()} | {IMAGE_SIZE}px NGF={NGF} T={T_DIFF} "
      f"σ=[{SIGMA_MIN},{SIGMA_MAX}] AMP={USE_AMP} GradCkpt={USE_GRAD_CKPT}")

# ══════════════════════════════════════════════════════════════════════════
# 1.  DATASET  (identical structure to Cell 7)
# ══════════════════════════════════════════════════════════════════════════
class BCIFolderDataset(Dataset):
    def __init__(self, split, image_size=IMAGE_SIZE, augment=True):
        self.dir_A = os.path.join(DATA_ROOT, split, "A")
        self.dir_B = os.path.join(DATA_ROOT, split, "B")
        self.files = sorted(f for f in os.listdir(self.dir_A) if f.endswith(".png"))
        self.augment = augment; self.size = image_size
        print(f"  [{split:5s}] {len(self.files)} pairs")

    def __len__(self): return len(self.files)

    def _to_tensor(self, img):
        return T.ToTensor()(img.resize((self.size, self.size), Image.BICUBIC)) * 2 - 1

    def __getitem__(self, idx):
        fname = self.files[idx]
        he  = Image.open(os.path.join(self.dir_A, fname)).convert("RGB")
        ihc = Image.open(os.path.join(self.dir_B, fname.replace("_A.", "_B."))).convert("RGB")
        if self.augment:
            if random.random() > 0.5: he, ihc = TF.hflip(he), TF.hflip(ihc)
            if random.random() > 0.5: he, ihc = TF.vflip(he), TF.vflip(ihc)
        return self._to_tensor(he), self._to_tensor(ihc)

print("\nLoading datasets...")
train_ds = BCIFolderDataset("train", augment=True)
val_ds   = BCIFolderDataset("val",   augment=False)

_pin     = (MODE == "newton")
_persist = (MODE == "newton")
_nw      = 2 if MODE == "colab" else 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=_nw, pin_memory=_pin, drop_last=True,
                          persistent_workers=_persist)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=_nw, pin_memory=_pin,
                          persistent_workers=_persist)

iters_per_epoch = len(train_loader)
total_opt_steps = max(1, (N_EPOCHS * iters_per_epoch) // max(ACCUM, 1))
print(f"Train iters/epoch : {iters_per_epoch} | Total opt steps : {total_opt_steps}")

# ══════════════════════════════════════════════════════════════════════════
# 2.  SCORE-BASED NOISE SCHEDULE
#
#  Geometric (NCSN-style) noise schedule — key difference from BBDM / PST-Diff.
#  σ_i decreases from σ_max → σ_min over T steps (index 0 = most noisy).
#
#  Perturb : x_t = x_0 + σ_t · ε          (additive, not VP like DDPM)
#  Score   : s* = -ε / σ_t                 (gradient of log p)
#  Loss    : σ_t² · ‖s_θ + ε/σ_t‖²        (σ² weighting balances scale)
#          = ‖σ_t · s_θ + ε‖²             (what we compute in practice)
#
#  At inference: annealed Langevin dynamics (predictor-corrector).
# ══════════════════════════════════════════════════════════════════════════
class ScoreSchedule:
    def __init__(self, T=T_DIFF, s_min=SIGMA_MIN, s_max=SIGMA_MAX, device=DEVICE):
        self.T      = T
        self.sigmas = torch.exp(
            torch.linspace(math.log(s_max), math.log(s_min), T, device=device)
        )  # sigmas[0]=σ_max (most noisy), sigmas[-1]=σ_min

    def perturb(self, x0, t):
        """x_t = x0 + σ_t · ε"""
        sigma = self.sigmas[t].view(-1, 1, 1, 1)
        noise = torch.randn_like(x0)
        return x0 + sigma * noise, noise, sigma

    @torch.no_grad()
    def sample(self, score_fn, shape, ddim_steps=None, device=DEVICE):
        """
        Annealed Langevin dynamics with predictor (Euler-Maruyama) step.
        ddim_steps controls how many noise levels to skip (subsampling).
        """
        steps = ddim_steps or self.T
        step_idx = list(range(0, self.T, max(1, self.T // steps)))

        x = torch.randn(shape, device=device) * self.sigmas[0]
        eps_ref = self.sigmas[-1]   # σ_min as reference

        for i in step_idx:
            sigma     = self.sigmas[i]
            sigma_next = (self.sigmas[i + max(1, self.T // steps)]
                          if i + max(1, self.T // steps) < self.T
                          else self.sigmas[-1])

            t_b = torch.full((shape[0],), i, device=device, dtype=torch.long)

            # Langevin corrector (1 step)
            alpha_c = 0.1 * (sigma / eps_ref) ** 2
            grad    = score_fn(x, t_b)
            z       = torch.randn_like(x)
            x       = x + alpha_c * grad + math.sqrt(2 * alpha_c) * z

            # Predictor (Euler-Maruyama)
            grad = score_fn(x, t_b)
            x    = x + (sigma ** 2 - sigma_next ** 2) * grad

        return x.clamp(-1, 1)

schedule = ScoreSchedule()

# ══════════════════════════════════════════════════════════════════════════
# 3.  ARCHITECTURE
# ══════════════════════════════════════════════════════════════════════════

def _flash_attn(q, k, v):
    orig = q.dtype
    # .contiguous() ensures stride-1 on last dim — required by all fused kernels
    q = q.half().contiguous()
    k = k.half().contiguous()
    v = v.half().contiguous()
    out = F.scaled_dot_product_attention(q, k, v)
    return out.to(orig)


class TimeEmb(nn.Module):
    """Sinusoidal timestep embedding — maps σ index to continuous vector."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim*4), nn.SiLU(),
                                  nn.Linear(dim*4, dim*4))

    def forward(self, t):
        half = self.dim // 2
        freq = torch.exp(-math.log(10000) *
                         torch.arange(half, device=t.device) / max(half-1, 1))
        emb  = t.float()[:, None] * freq[None]
        return self.mlp(torch.cat([emb.sin(), emb.cos()], dim=-1))


class ResBlock(nn.Module):
    def __init__(self, ic, oc, td):
        super().__init__()
        self.n1 = nn.GroupNorm(min(8, ic), ic); self.c1 = nn.Conv2d(ic, oc, 3,1,1)
        self.tp = nn.Linear(td, oc*2)
        self.n2 = nn.GroupNorm(min(8, oc), oc); self.c2 = nn.Conv2d(oc, oc, 3,1,1)
        self.sk = nn.Conv2d(ic, oc, 1) if ic != oc else nn.Identity()
        nn.init.zeros_(self.c2.weight); nn.init.zeros_(self.c2.bias)

    def forward(self, x, te):
        h = self.c1(F.silu(self.n1(x)))
        t = self.tp(F.silu(te))[:, :, None, None]; sc, sh = t.chunk(2, dim=1)
        return self.c2(F.silu(self.n2(h)*(1+sc)+sh)) + self.sk(x)


class SelfAttn(nn.Module):
    """Flash Attention self-attention."""
    def __init__(self, ch, heads=4):
        super().__init__()
        self.heads = heads
        self.norm  = nn.GroupNorm(min(8, ch), ch)
        self.qkv   = nn.Conv2d(ch, ch*3, 1)
        self.proj  = nn.Conv2d(ch, ch, 1)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        B, C, H, W = x.shape; d = C // self.heads
        q, k, v = self.qkv(self.norm(x)).chunk(3, dim=1)
        def re(t): return t.view(B, self.heads, d, H*W).transpose(2, 3)
        return x + self.proj(_flash_attn(re(q),re(k),re(v)).transpose(2,3).reshape(B,C,H,W))


class CrossAttn(nn.Module):
    """
    Standard cross-attention: Q from noisy UNet state, K/V from H&E context.
    Simpler than PST-Diff's AAM — no asymmetric gate, no LT module.
    This is the primary architectural distinction from Cell 7.
    """
    def __init__(self, ch, ctx_ch, heads=4):
        super().__init__()
        self.heads   = heads
        self.norm_q  = nn.GroupNorm(min(8, ch), ch)
        self.norm_kv = nn.GroupNorm(min(8, ctx_ch), ctx_ch)
        self.q       = nn.Conv2d(ch, ch, 1)
        self.k       = nn.Conv2d(ctx_ch, ch, 1)
        self.v       = nn.Conv2d(ctx_ch, ch, 1)
        self.proj    = nn.Conv2d(ch, ch, 1)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)

    def forward(self, x, ctx):
        B, C, H, W = x.shape
        # Keep ctx at its native resolution — asymmetric Q/K is correct and cheap
        # Do NOT interpolate ctx up to (H,W) — that causes O(N²) explosion
        Hc, Wc = ctx.shape[-2], ctx.shape[-1]
        q = self.q(self.norm_q(x))
        k = self.k(self.norm_kv(ctx))
        v = self.v(ctx)
        d = C // self.heads
        def re_q(t):  return t.view(B, self.heads, d, H*W).transpose(2, 3)
        def re_kv(t): return t.view(B, self.heads, d, Hc*Wc).transpose(2, 3)
        out = _flash_attn(re_q(q), re_kv(k), re_kv(v))
        out = out.transpose(2, 3).reshape(B, C, H, W)
        return x + self.proj(out)


class HEEncoder(nn.Module):
    """Multi-scale H&E feature extractor (same as Cell 7 for fair comparison)."""
    def __init__(self, ngf=NGF):
        super().__init__()
        self.e1 = nn.Sequential(nn.Conv2d(3,ngf,3,1,1), nn.GroupNorm(min(8,ngf),ngf), nn.SiLU())
        self.e2 = nn.Sequential(nn.Conv2d(ngf,ngf*2,3,2,1), nn.GroupNorm(min(8,ngf*2),ngf*2), nn.SiLU())
        self.e3 = nn.Sequential(nn.Conv2d(ngf*2,ngf*4,3,2,1), nn.GroupNorm(min(8,ngf*4),ngf*4), nn.SiLU())
        self.e4 = nn.Sequential(nn.Conv2d(ngf*4,ngf*4,3,2,1), nn.GroupNorm(min(8,ngf*4),ngf*4), nn.SiLU())

    def forward(self, x):
        f1=self.e1(x); f2=self.e2(f1); f3=self.e3(f2); f4=self.e4(f3)
        return f1, f2, f3, f4


class ScoreUNet(nn.Module):
    """
    Score function UNet conditioned on H&E via cross-attention.

    Outputs raw network activations; the caller applies score scaling:
        score s_θ = output / (-σ_t)

    Cross-attention at bottleneck + decoder (replaces AAM+LT from PST-Diff).
    Gradient checkpointing on ResBlocks when use_ckpt=True.

    Also optionally returns decoder hidden features at full resolution
    for the MI contrastive loss (return_dec_feat=True).
    """
    def __init__(self, ngf=NGF, t_mult=4, use_ckpt=USE_GRAD_CKPT):
        super().__init__()
        self.use_ckpt = use_ckpt
        td = ngf * t_mult
        ch = [ngf, ngf*2, ngf*4, ngf*4]

        self.time_emb = TimeEmb(ngf)
        self.in_conv  = nn.Conv2d(3, ch[0], 3,1,1)

        # Encoder
        self.e1 = nn.ModuleList([ResBlock(ch[0],ch[0],td), ResBlock(ch[0],ch[0],td)])
        self.d1 = nn.Conv2d(ch[0], ch[0], 3,2,1)
        self.e2 = nn.ModuleList([ResBlock(ch[0],ch[1],td), ResBlock(ch[1],ch[1],td)])
        self.d2 = nn.Conv2d(ch[1], ch[1], 3,2,1)
        self.e3 = nn.ModuleList([ResBlock(ch[1],ch[2],td), ResBlock(ch[2],ch[2],td), SelfAttn(ch[2])])
        self.d3 = nn.Conv2d(ch[2], ch[2], 3,2,1)
        self.e4 = nn.ModuleList([ResBlock(ch[2],ch[3],td), ResBlock(ch[3],ch[3],td), SelfAttn(ch[3])])

        # Bottleneck + cross-attention on H&E f4
        self.bot1    = ResBlock(ch[3], ch[3], td)
        self.bot_ca  = CrossAttn(ch[3], ch[3])    # ← cross-attn on H&E f4
        self.bot2    = ResBlock(ch[3], ch[3], td)

        # Decoder with cross-attention on H&E features
        self.u3 = nn.ConvTranspose2d(ch[3], ch[2], 2,2)
        self.ca3 = CrossAttn(ch[2], ch[3])         # ← cross-attn on H&E f4
        self.r3 = nn.ModuleList([ResBlock(ch[2]*2,ch[2],td), ResBlock(ch[2],ch[2],td), SelfAttn(ch[2])])

        self.u2 = nn.ConvTranspose2d(ch[2], ch[1], 2,2)
        self.ca2 = CrossAttn(ch[1], ch[2])         # ← cross-attn on H&E f3
        self.r2 = nn.ModuleList([ResBlock(ch[1]*2,ch[1],td), ResBlock(ch[1],ch[1],td)])

        self.u1 = nn.ConvTranspose2d(ch[1], ch[0], 2,2)
        self.ca1 = CrossAttn(ch[0], ch[1])         # ← cross-attn on H&E f2
        self.r1 = nn.ModuleList([ResBlock(ch[0]*2,ch[0],td), ResBlock(ch[0],ch[0],td)])

        self.out_norm = nn.GroupNorm(min(8,ch[0]), ch[0])
        self.out_conv = nn.Conv2d(ch[0], 3, 3,1,1)
        nn.init.zeros_(self.out_conv.weight); nn.init.zeros_(self.out_conv.bias)

    def _run(self, x, blocks, te):
        for b in blocks:
            if isinstance(b, SelfAttn): x = b(x)
            elif self.use_ckpt and self.training:
                x = grad_checkpoint(b, x, te, use_reentrant=False)
            else: x = b(x, te)
        return x

    def forward(self, x_t, t, he_feats, return_dec_feat=False):
        f1, f2, f3, f4 = he_feats
        te = self.time_emb(t)

        # Encoder
        h  = self.in_conv(x_t)
        h1 = self._run(h,            self.e1, te)
        h2 = self._run(self.d1(h1),  self.e2, te)
        h3 = self._run(self.d2(h2),  self.e3, te)
        h4 = self._run(self.d3(h3),  self.e4, te)

        # Bottleneck + cross-attn
        b  = self.bot1(h4, te)
        b  = self.bot_ca(b, f4)
        b  = self.bot2(b, te)

        # Decoder
        d  = self.ca3(self.u3(b), f4)
        d  = self._run(torch.cat([d, h3], 1), self.r3, te)

        d  = self.ca2(self.u2(d), f3)
        d  = self._run(torch.cat([d, h2], 1), self.r2, te)

        d  = self.ca1(self.u1(d), f2)
        d  = self._run(torch.cat([d, h1], 1), self.r1, te)

        dec_feat = d  # [B, NGF, H, W] — full resolution, for MI loss

        out = self.out_conv(F.silu(self.out_norm(d)))
        if return_dec_feat:
            return out, dec_feat
        return out


# ══════════════════════════════════════════════════════════════════════════
# 4.  MI PROJECTION HEADS  (lightweight 1×1 convs to shared embedding space)
#     Projects H&E encoder features (f1) and ScoreUNet decoder features
#     to the same channel dim for InfoNCE comparison.
# ══════════════════════════════════════════════════════════════════════════
class MIProjector(nn.Module):
    def __init__(self, in_ch, feat_dim=64):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_ch, feat_dim, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(feat_dim, feat_dim, 1, bias=False))

    def forward(self, x): return self.proj(x)


# ══════════════════════════════════════════════════════════════════════════
# 5.  LOSSES
# ══════════════════════════════════════════════════════════════════════════

# ── 5a. Score matching loss (SNR-weighted) ────────────────────────────────
def score_matching_loss(output, noise, sigma):
    """
    σ² · ‖output/(-σ) - (-ε/σ)‖² = ‖output + ε‖² / σ²  ... simplifies to:
    σ² weighting: ‖σ · s_θ + ε‖² where s_θ = output / σ
    In practice: weight * L2(output/σ, -ε/σ) = L2(output + ε) * (1/σ²) * σ²

    Net result: loss = mean( (output + noise)² )
    This is equivalent to denoising score matching with σ² importance weight.
    The SNR weighting means the model is penalised most at low-noise steps
    (where σ is small), unlike DDPM which weights all timesteps equally.
    """
    return ((output + noise) ** 2).mean()


# ── 5b. Topology Consistency Loss (soft Euler characteristic) ─────────────
def _soft_euler(x, tau, beta=20.0):
    s  = torch.sigmoid(beta * (x - tau)).mean(dim=1, keepdim=True)
    H, W = s.shape[-2], s.shape[-1]
    V  = s.sum(dim=(-2,-1))
    Eh = (s[..., :, :-1] * s[..., :,  1:]).sum(dim=(-2,-1))
    Ev = (s[..., :-1, :] * s[..., 1:,  :]).sum(dim=(-2,-1))
    Fs = (s[..., :-1, :-1] * s[..., :-1, 1:] *
          s[..., 1:,  :-1] * s[..., 1:,  1:]).sum(dim=(-2,-1))
    return (V - (Eh + Ev) + Fs) / (H * W)   # ← normalize to [0,1] range


def topology_loss(pred, target, n_thresh=N_TOPO_THRESH, beta=20.0):
    thresholds = torch.linspace(-0.8, 0.8, n_thresh, device=pred.device)
    loss = torch.tensor(0.0, device=pred.device)
    for tau in thresholds:
        chi_p = _soft_euler(pred,            tau, beta)
        chi_t = _soft_euler(target.detach(), tau, beta)
        loss  = loss + F.l1_loss(chi_p, chi_t)   # ← L1 not MSE
    return loss / n_thresh


# ── 5c. MI Contrastive Loss (InfoNCE) ─────────────────────────────────────
def mi_loss(q_proj, k_proj, n_patches=N_MI_PATCHES, temperature=0.07):
    """
    InfoNCE between H&E projected features (k) and predicted-IHC features (q).
    Maximises mutual information between modalities at corresponding patches,
    encouraging the score network to preserve H&E morphological structure
    in the IHC prediction.
    Motivated by MIU-Diff (arxiv 2506.23184).
    """
    B, C, H, W = q_proj.shape
    if k_proj.shape[-2:] != (H, W):
        k_proj = F.interpolate(k_proj, (H, W), mode="bilinear", align_corners=False)

    N = H * W
    n = min(n_patches, N)
    idx = torch.randperm(N, device=q_proj.device)[:n]

    # [B, n, C]
    q = F.normalize(q_proj.view(B, C, N).permute(0,2,1)[:, idx], dim=-1)
    k = F.normalize(k_proj.view(B, C, N).permute(0,2,1)[:, idx], dim=-1)

    logits = torch.bmm(q, k.transpose(1, 2)) / temperature   # [B, n, n]
    labels = torch.arange(n, device=q_proj.device).expand(B, -1)
    return F.cross_entropy(logits.reshape(B*n, n), labels.reshape(-1))


# ══════════════════════════════════════════════════════════════════════════
# 6.  MODEL ASSEMBLY
# ══════════════════════════════════════════════════════════════════════════
MI_FEAT_DIM = max(32, NGF)   # shared MI embedding dimension

print(f"\nBuilding Score-Based Diffusion + Topology  [{MODE.upper()} mode]...")
he_enc     = HEEncoder(ngf=NGF).to(DEVICE)
score_unet = ScoreUNet(ngf=NGF, use_ckpt=USE_GRAD_CKPT).to(DEVICE)
mi_proj_he = MIProjector(in_ch=NGF,       feat_dim=MI_FEAT_DIM).to(DEVICE)
mi_proj_ihc= MIProjector(in_ch=NGF,       feat_dim=MI_FEAT_DIM).to(DEVICE)

def cp(m): return sum(p.numel() for p in m.parameters()) / 1e6
print(f"  HEEncoder   : {cp(he_enc):.2f}M")
print(f"  ScoreUNet   : {cp(score_unet):.2f}M")
print(f"  MI Proj (×2): {cp(mi_proj_he)+cp(mi_proj_ihc):.2f}M")
print(f"  Total       : {cp(he_enc)+cp(score_unet)+cp(mi_proj_he)+cp(mi_proj_ihc):.2f}M")

all_params = (list(he_enc.parameters()) + list(score_unet.parameters()) +
              list(mi_proj_he.parameters()) + list(mi_proj_ihc.parameters()))
opt    = AdamW(all_params, lr=LR, betas=(0.9, 0.999), weight_decay=1e-4)
sched  = CosineAnnealingLR(opt, T_max=total_opt_steps, eta_min=LR*0.01)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# ══════════════════════════════════════════════════════════════════════════
# 7.  METRICS
# ══════════════════════════════════════════════════════════════════════════
def psnr(a, b):
    return 10 * math.log10(4.0 / (((a.clamp(-1,1)-b.clamp(-1,1))**2).mean().item() + 1e-8))

def ssim_fast(a, b):
    k, p = 11, 5
    ma, mb = F.avg_pool2d(a,k,1,p), F.avg_pool2d(b,k,1,p)
    sa = F.avg_pool2d(a**2,k,1,p)-ma**2; sb = F.avg_pool2d(b**2,k,1,p)-mb**2
    sab= F.avg_pool2d(a*b, k,1,p)-ma*mb; C1,C2 = 0.01**2, 0.03**2
    return (((2*ma*mb+C1)*(2*sab+C2))/((ma**2+mb**2+C1)*(sa+sb+C2)).clamp_min(1e-8)).mean().item()

# ══════════════════════════════════════════════════════════════════════════
# 8.  CSV HELPERS  (columns match watch_metrics.py / Cell 7 for direct comparison)
# ══════════════════════════════════════════════════════════════════════════
def init_csv(p, f):
    with open(p,"w",newline="",buffering=1) as fp: csv.writer(fp).writerow(f)

def append_csv(p, r):
    with open(p,"a",newline="",buffering=1) as fp: csv.writer(fp).writerow(r)

init_csv(LOG_CSV,   ["wall_time","epoch","iter","global_step",
                     "loss_total","loss_score","loss_topo","loss_mi"])
init_csv(EPOCH_CSV, ["wall_time","epoch","mean_loss_total",
                     "mean_loss_score","mean_loss_topo","mean_loss_mi",
                     "val_psnr","val_ssim"])

# ══════════════════════════════════════════════════════════════════════════
# 9.  SAMPLE SAVER  (recon/ + target/ dirs — matches evaluate.py convention)
# ══════════════════════════════════════════════════════════════════════════
def save_samples(epoch, he, x0_pred, real_ihc, global_step=0, tag="epoch"):
    def norm(t): return ((t.detach().cpu().clamp(-1,1)+1)/2)
    imgs  = torch.cat([norm(he), norm(x0_pred), norm(real_ihc)], dim=0)
    grid  = vutils.make_grid(imgs, nrow=he.shape[0], padding=2, normalize=False)
    fname = os.path.join(SAMPLE_DIR, f"{tag}_ep{epoch:04d}_step{global_step:07d}.png")
    vutils.save_image(grid, fname)
    print(f"  [Sample] {fname}")


# ══════════════════════════════════════════════════════════════════════════
# 10.  VALIDATION
# ══════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def validate(epoch, max_batches=MAX_VAL_BATCHES, ddim_steps=VAL_DDIM_STEPS):
    he_enc.eval(); score_unet.eval()
    psnr_s, ssim_s = [], []

    for i, (he, ihc) in enumerate(val_loader):
        if i >= max_batches: break
        he, ihc = he.to(DEVICE, non_blocking=True), ihc.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            feats = he_enc(he)
            # Score function closure for sampler
            def score_fn(x, t_b):
                out = score_unet(x, t_b, feats)
                sigma = schedule.sigmas[t_b].view(-1,1,1,1)
                return out / (-sigma)   # s_θ = network_out / (-σ)

            fake = schedule.sample(score_fn, he.shape, ddim_steps=ddim_steps, device=DEVICE)

        psnr_s.append(psnr(fake, ihc))
        ssim_s.append(ssim_fast(fake, ihc))
        torch.cuda.empty_cache()

    he_enc.train(); score_unet.train()
    return float(np.mean(psnr_s)), float(np.mean(ssim_s))


def backup(epoch, ckpt):
    import shutil
    try:
        shutil.copy2(ckpt, os.path.join(DRIVE_CKPT, os.path.basename(ckpt)))
        print(f"  [Backup] saved")
    except Exception as e:
        print(f"  [Backup] failed (non-fatal): {e}")


# ══════════════════════════════════════════════════════════════════════════
# 11.  TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"  Score-Based Diffusion + Topology  [{MODE.upper()}]")
print(f"  {IMAGE_SIZE}px  NGF={NGF}  T={T_DIFF}  σ=[{SIGMA_MIN},{SIGMA_MAX}]")
print(f"  Epochs={N_EPOCHS}  EffBatch={BATCH_SIZE*ACCUM}")
print(f"  Losses: score(SNR-weighted) + topo(χ-Euler,{N_TOPO_THRESH}thresh) + MI(InfoNCE)")
print("="*60 + "\n")

global_step  = 0
opt_step     = 0
wall_start   = time.time()
timeout_flag = False

last_val_psnr = 0.0
last_val_ssim = 0.0
last_epoch    = 0

for epoch in range(1, N_EPOCHS+1):
    if timeout_flag: break

    ep = {"total":[], "score":[], "topo":[], "mi":[]}
    opt.zero_grad()

    _vh, _vi = next(iter(val_loader))
    _vh, _vi = _vh.to(DEVICE), _vi.to(DEVICE)

    for it, (real_he, real_ihc) in enumerate(train_loader):
        global_step += 1
        real_he  = real_he.to(DEVICE,  non_blocking=True)
        real_ihc = real_ihc.to(DEVICE, non_blocking=True)

        # Random noise level index
        t = torch.randint(0, T_DIFF, (real_he.shape[0],), device=DEVICE)

        # ── 1. Encode H&E ───────────────────────────────────────────────
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            he_feats = he_enc(real_he)

        # ── 2. Perturb IHC with geometric noise ─────────────────────────
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            x_t, noise, sigma = schedule.perturb(real_ihc, t)

        # ── 3. Score network forward (returns decoder feat for MI) ───────
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            net_out, dec_feat = score_unet(x_t, t, he_feats, return_dec_feat=True)

            # Score: s_θ = net_out / (-σ)
            # Score loss: ‖net_out + ε‖² (σ²-weighted, see docstring above)
            l_score = score_matching_loss(net_out, noise, sigma)

            # Recover x0 estimate for topology loss
            # x0_pred = x_t + σ² · s_θ = x_t - σ · net_out
            x0_pred = (x_t - sigma * net_out).clamp(-1, 1)

            # ── 4. Topology loss ─────────────────────────────────────────
            l_topo = topology_loss(x0_pred, real_ihc, n_thresh=N_TOPO_THRESH)

            # ── 5. MI contrastive loss ───────────────────────────────────
            q_feat = mi_proj_ihc(dec_feat)           # predicted IHC features
            k_feat = mi_proj_he(he_feats[0].detach())# H&E f1 features (detached)
            l_mi   = mi_loss(q_feat, k_feat, n_patches=N_MI_PATCHES)

            loss = l_score + LAMBDA_TOPO * l_topo + LAMBDA_MI * l_mi

        # ── 6. Backward + grad accumulation ─────────────────────────────
        scaler.scale(loss / max(ACCUM, 1)).backward()

        if global_step % max(ACCUM, 1) == 0:
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(all_params, 1.0)
            scaler.step(opt); scaler.update()
            sched.step(); opt.zero_grad()
            opt_step += 1

        for k, v in zip(["total","score","topo","mi"],[loss,l_score,l_topo,l_mi]):
            ep[k].append(v.item())

        # ── Logging ──────────────────────────────────────────────────────
        if global_step % SAVE_EVERY == 0:
            ts  = datetime.now().isoformat(timespec="seconds")
            elm = (time.time()-wall_start)/60
            append_csv(LOG_CSV, [ts, epoch, it+1, global_step,
                                  f"{loss.item():.6f}", f"{l_score.item():.6f}",
                                  f"{l_topo.item():.6f}", f"{l_mi.item():.6f}"])
            print(f"[{ts}] Ep{epoch}/{N_EPOCHS} It{it+1} {elm:.1f}m | "
                  f"loss={loss.item():.4f} score={l_score.item():.4f} "
                  f"topo={l_topo.item():.4f} mi={l_mi.item():.4f}")

        if global_step % SAMPLE_EVERY == 0:
            save_samples(epoch, real_he, x0_pred, real_ihc,
                         global_step=global_step, tag="iter")

        # was:   if MODE == "colab" and time.time()-wall_start > MAX_WALL_SECS:
        if time.time()-wall_start > MAX_WALL_SECS:
            print(f"\n[TIMEOUT] Wall cap reached at epoch {epoch} iter {it+1}.")
            timeout_flag = True; break

    # ── Epoch end ─────────────────────────────────────────────────────────
    last_epoch = epoch
    ml = {k: float(np.mean(v)) for k, v in ep.items()}
    torch.cuda.empty_cache(); gc.collect()

    if epoch % CKPT_EVERY_N == 0 or epoch in EXPLICIT_CKPT_EPOCHS or timeout_flag:
        val_psnr, val_ssim = validate(epoch)
        last_val_psnr = val_psnr; last_val_ssim = val_ssim

        ts = datetime.now().isoformat(timespec="seconds")
        append_csv(EPOCH_CSV, [ts, epoch,
                                f"{ml['total']:.6f}", f"{ml['score']:.6f}",
                                f"{ml['topo']:.6f}",  f"{ml['mi']:.6f}",
                                f"{val_psnr:.4f}",    f"{val_ssim:.6f}"])

        ckpt = os.path.join(CKPT_DIR, f"epoch_{epoch:04d}.pt")
        torch.save({
            "epoch": epoch, "opt_step": opt_step, "mode": MODE,
            "he_enc":      he_enc.state_dict(),
            "score_unet":  score_unet.state_dict(),
            "mi_proj_he":  mi_proj_he.state_dict(),
            "mi_proj_ihc": mi_proj_ihc.state_dict(),
            "opt": opt.state_dict(), "sched": sched.state_dict(),
            "scaler": scaler.state_dict(),
            "val_psnr": val_psnr, "val_ssim": val_ssim,
        }, ckpt)

        if epoch in EXPLICIT_CKPT_EPOCHS:
            import shutil
            shutil.copy2(ckpt, ckpt.replace(".pt", "_explicit.pt"))
            print(f"  [Ckpt] Explicit epoch {epoch} saved")

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                _feats = he_enc(_vh)
                def _score_fn(x, t_b):
                    out = score_unet(x, t_b, _feats)
                    sig = schedule.sigmas[t_b].view(-1,1,1,1)
                    return out / (-sig)
                _sample = schedule.sample(_score_fn, _vi.shape,
                                           ddim_steps=VAL_DDIM_STEPS, device=DEVICE)
        save_samples(epoch, _vh, _sample, _vi, global_step=global_step, tag="epoch")
        backup(epoch, ckpt)

    elapsed = (time.time()-wall_start)/60
    print(f"\n{'='*60}")
    print(f"  Epoch {epoch}/{N_EPOCHS} [{MODE.upper()}] | {elapsed:.1f}min")
    print(f"  Loss  total={ml['total']:.4f}  score={ml['score']:.4f}  "
          f"topo={ml['topo']:.4f}  mi={ml['mi']:.4f}")
    if epoch % CKPT_EVERY_N == 0 or epoch in EXPLICIT_CKPT_EPOCHS:
        print(f"  Val   PSNR={val_psnr:.4f} dB  SSIM={val_ssim:.6f}")
    print(f"{'='*60}\n")

    if timeout_flag: break

if timeout_flag and last_epoch > 0:
    import shutil
    final = os.path.join(CKPT_DIR, f"final_epoch{last_epoch}.pt")
    torch.save({"epoch": last_epoch, "he_enc": he_enc.state_dict(),
                "score_unet": score_unet.state_dict(),
                "val_psnr": last_val_psnr, "val_ssim": last_val_ssim}, final)
    shutil.copy2(final, os.path.join(DRIVE_CKPT, os.path.basename(final)))
    print(f"  [Ckpt] Final checkpoint → {final}")

print("\nTraining complete.")
print(f"  Mode     : {MODE.upper()}")
print(f"  Iter CSV : {LOG_CSV}")
print(f"  Epoch CSV: {EPOCH_CSV}")
print(f"  Ckpts    : {CKPT_DIR}")
print(f"  Samples  : {SAMPLE_DIR}")

GPU memory before build: 10.49 GiB  (should be near 0)
Device : cuda | GPU : NVIDIA A100-SXM4-40GB
Mode   : A100 | 256px NGF=128 T=1000 σ=[0.01,50.0] AMP=True GradCkpt=True

Loading datasets...
  [train] 3311 pairs
  [val  ] 585 pairs
Train iters/epoch : 827 | Total opt steps : 20675

Building Score-Based Diffusion + Topology  [A100 mode]...
  HEEncoder   : 3.84M
  ScoreUNet   : 62.83M
  MI Proj (×2): 0.07M
  Total       : 66.74M

  Score-Based Diffusion + Topology  [A100]
  256px  NGF=128  T=1000  σ=[0.01,50.0]
  Epochs=50  EffBatch=8
  Losses: score(SNR-weighted) + topo(χ-Euler,10thresh) + MI(InfoNCE)

[2026-05-06T23:36:05] Ep1/50 It100 0.5m | loss=0.5167 score=0.3017 topo=0.0571 mi=4.1862


KeyboardInterrupt: 

Cell 10: HistDIT + SAM Conditioning

In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — HistDiT + SAM Conditioning                                   ║
# ║  [DUAL MODE: colab (T4 verify) / a100 (A100 actual training)]          ║
# ║                                                                          ║
# ║  Key architectural distinctions from Cells 7 & 8:                      ║
# ║   1. DiT (Diffusion Transformer) backbone — NOT a UNet                 ║
# ║      Patches H&E noisy IHC image → Transformer blocks → unpatch        ║
# ║   2. adaLN-Zero conditioning (DiT paper convention)                    ║
# ║      Time step + SAM global features condition via adaptive LayerNorm   ║
# ║   3. Dual-stream SAM conditioning                                       ║
# ║      Stream A: SAM global pooled features → adaLN scale/shift          ║
# ║      Stream B: SAM spatial features → cross-attention per DiT block     ║
# ║   4. L2 noise prediction loss (DiT standard, vs L1 in Cell 7)          ║
# ║   5. SAM Proxy (colab) or frozen SAM ViT-B (a100)                      ║
# ║                                                                          ║
# ║  Sources: HistDiT (arxiv 2604.08305), SAM conditioning (#25 in list),  ║
# ║           DiT (Peebles & Xie, ICCV 2023)                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ▶▶  SET THIS BEFORE RUNNING  ◀◀
#
#  "colab"  → T4 GPU, SAM proxy CNN, tiny DiT, 64px  — PIPELINE VERIFY ONLY
#  "a100"   → A100 GPU, real SAM ViT-B frozen, full DiT, 256px — ACTUAL RUN
#
MODE = "a100"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Cleanup: release stale allocations from prior runs ─────────────────────
import gc, torch
for _var in ['he_enc', 'sam_encoder', 'dit', 'opt', 'sched', 'scaler',
             'train_loader', 'val_loader']:
    if _var in globals(): del globals()[_var]
torch.cuda.empty_cache(); gc.collect()
_mem = torch.cuda.memory_allocated() / 1024**3
print(f"GPU memory before build: {_mem:.2f} GiB  (should be near 0)")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")

assert MODE in ("colab", "a100"), "MODE must be 'colab' or 'a100'"

# ── Imports ────────────────────────────────────────────────────────────────
import os, csv, time, math, random, shutil
import numpy as np
from datetime import datetime
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.checkpoint import checkpoint as grad_checkpoint

# ── Environment ────────────────────────────────────────────────────────────
torch.cuda.empty_cache(); gc.collect()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════
if MODE == "colab":
    # ── T4 pipeline verification — tiny everything ──────────────────────
    IMAGE_SIZE      = 64
    PATCH_SIZE      = 8        # N = (64/8)² = 64 tokens
    DIT_DIM         = 128      # transformer hidden dim
    DIT_DEPTH       = 4        # number of DiT blocks
    DIT_HEADS       = 4        # attention heads in DiT
    NGF             = 64       # HE encoder base channels
    SAM_FEAT_DIM    = 64       # SAM proxy output channels
    BATCH_SIZE      = 1
    ACCUM           = 2
    T_DIFF          = 100
    N_EPOCHS        = 2        # enough to see the pipeline runs
    LR              = 1e-4
    BETA_START      = 1e-4; BETA_END = 0.02
    LAMBDA_SAM      = 0.1      # SAM structural loss weight
    SAVE_EVERY      = 50
    SAMPLE_EVERY    = 200
    MAX_VAL_BATCHES = 4
    VAL_DDIM_STEPS  = 10
    USE_AMP         = True
    USE_GRAD_CKPT   = True
    USE_REAL_SAM    = False    # SAM proxy CNN — no download needed
    MAX_WALL_SECS   = 4*3600 - 300
    EXPLICIT_CKPT_EPOCHS = {8}
    RESULTS_DIR     = "/content/results/HistDiT_SAM_COLAB"
    DRIVE_CKPT      = "/content/drive/MyDrive/HistDiT_SAM_checkpoints_colab"

else:  # a100
    IMAGE_SIZE      = 512    # was 256 — matches all other models, 16× more DiT compute
    PATCH_SIZE      = 16     # N = (512/16)² = 1024 tokens
    DIT_DIM         = 512
    DIT_DEPTH       = 12
    DIT_HEADS       = 8
    NGF             = 64
    SAM_FEAT_DIM    = 256
    BATCH_SIZE      = 16      # conservative at 512px; increase to 16 if GPU RAM stays under 60GB
    ACCUM           = 2      # eff batch = 16
    T_DIFF          = 1000
    N_EPOCHS        = 50    # wall clock controls stop, not epoch count
    LR              = 1e-4
    BETA_START      = 1e-4; BETA_END = 0.02
    LAMBDA_SAM      = 0.1
    SAVE_EVERY      = 100
    SAMPLE_EVERY    = 500
    MAX_VAL_BATCHES = 16
    VAL_DDIM_STEPS  = 50
    USE_AMP         = True
    USE_GRAD_CKPT   = False
    USE_REAL_SAM    = False
    MAX_WALL_SECS   = 2*3600   # 2h55m — saves final ckpt before 3h cutoff
    EXPLICIT_CKPT_EPOCHS = {8}
    RESULTS_DIR     = "/content/results/HistDiT_SAM_A100"
    DRIVE_CKPT      = "/content/drive/MyDrive/HistDiT_SAM_checkpoints_a100"

AMP_DTYPE = torch.bfloat16   # A100 native dtype — faster than float16
CKPT_EVERY_N = 10   # save / validate every N epochs
DATA_ROOT  = "/content/datasets_standardized/BCI_512"
CKPT_DIR   = os.path.join(RESULTS_DIR, "checkpoints")
SAMPLE_DIR = os.path.join(RESULTS_DIR, "samples")
LOG_CSV    = os.path.join(RESULTS_DIR, "metrics_iter.csv")
EPOCH_CSV  = os.path.join(RESULTS_DIR, "metrics_epoch.csv")
for d in [CKPT_DIR, SAMPLE_DIR, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)

print(f"Device : {DEVICE} | GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Mode   : {MODE.upper()} | {IMAGE_SIZE}px | Patch={PATCH_SIZE} | "
      f"DiT={DIT_DIM}d×{DIT_DEPTH}L×{DIT_HEADS}h | SAM={'ViT-B' if USE_REAL_SAM else 'Proxy'}")

# ══════════════════════════════════════════════════════════════════════════
# 1.  SAM SETUP  (a100 mode only)
#
#  SAM ViT-B is loaded frozen and used as a pure feature extractor.
#  Its image encoder outputs [B, 256, 64, 64] for any input size.
#  We resize H&E to 1024×1024 for SAM (its native resolution),
#  extract embeddings, then pool/project for our DiT conditioning.
# ══════════════════════════════════════════════════════════════════════════
sam_image_encoder = None   # populated below if USE_REAL_SAM

if USE_REAL_SAM:
    print("\nSetting up SAM ViT-B...")
    try:
        from segment_anything import sam_model_registry
    except ImportError:
        print("  Installing segment-anything...")
        os.system("pip install -q git+https://github.com/facebookresearch/segment-anything.git")
        from segment_anything import sam_model_registry

    SAM_CKPT = "/content/sam_vit_b_01ec64.pth"
    if not os.path.exists(SAM_CKPT):
        print("  Downloading SAM ViT-B checkpoint (~375 MB)...")
        os.system(f"wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {SAM_CKPT}")

    _sam_full        = sam_model_registry["vit_b"](checkpoint=SAM_CKPT)
    sam_image_encoder = _sam_full.image_encoder.to(DEVICE).eval()
    for p in sam_image_encoder.parameters():
        p.requires_grad = False

    # SAM pixel normalisation constants
    SAM_PIXEL_MEAN = torch.tensor([123.675, 116.28, 103.53],
                                   device=DEVICE).view(1,3,1,1) / 255.0
    SAM_PIXEL_STD  = torch.tensor([58.395, 57.12, 57.375],
                                   device=DEVICE).view(1,3,1,1) / 255.0
    print(f"  SAM ViT-B loaded and frozen. "
          f"Params: {sum(p.numel() for p in sam_image_encoder.parameters())/1e6:.1f}M")

# ══════════════════════════════════════════════════════════════════════════
# 2.  DATASET  (identical to Cells 7 & 8)
# ══════════════════════════════════════════════════════════════════════════
class BCIFolderDataset(Dataset):
    def __init__(self, split, image_size=IMAGE_SIZE, augment=True,
                 explicit_files=None, force_dir=None):
        base_dir = force_dir or os.path.join(DATA_ROOT, split)
        self.dir_A   = os.path.join(base_dir, "A")
        self.dir_B   = os.path.join(base_dir, "B")
        self.files   = explicit_files or sorted(
            f for f in os.listdir(self.dir_A) if f.endswith(".png"))
        self.augment = augment; self.size = image_size
        print(f"  [{split:5s}] {len(self.files)} pairs")

    def __len__(self): return len(self.files)

    def _to_tensor(self, img):
        return T.ToTensor()(img.resize((self.size, self.size), Image.BICUBIC)) * 2 - 1

    def __getitem__(self, idx):
        fname = self.files[idx]
        he  = Image.open(os.path.join(self.dir_A, fname)).convert("RGB")
        ihc = Image.open(os.path.join(self.dir_B, fname.replace("_A.", "_B."))).convert("RGB")
        if self.augment:
            if random.random() > 0.5: he, ihc = TF.hflip(he), TF.hflip(ihc)
            if random.random() > 0.5: he, ihc = TF.vflip(he), TF.vflip(ihc)
        return self._to_tensor(he), self._to_tensor(ihc)

print("\nLoading datasets...")
_train_dir_A = os.path.join(DATA_ROOT, "train", "A")
_all_files   = sorted(f for f in os.listdir(_train_dir_A) if f.endswith(".png"))
_val_dir_A   = os.path.join(DATA_ROOT, "val", "A")

if os.path.exists(_val_dir_A):
    train_ds = BCIFolderDataset("train", augment=True)
    val_ds   = BCIFolderDataset("val",   augment=False)
else:
    # No val directory — carve last 15% of train as validation
    print("  [INFO] No val/ directory — using last 15% of train as val split")
    _n_val     = max(50, int(len(_all_files) * 0.15))
    _val_files = _all_files[-_n_val:]
    _trn_files = _all_files[:-_n_val]
    _train_dir = os.path.join(DATA_ROOT, "train")
    train_ds = BCIFolderDataset("train", augment=True, explicit_files=_trn_files, force_dir=_train_dir)
    val_ds   = BCIFolderDataset("val", augment=False, explicit_files=_val_files, force_dir=_train_dir)

_pin     = (MODE in ("newton", "a100"))
_persist = (MODE in ("newton", "a100"))
_nw      = 2 if MODE == "colab" else 4
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=_nw, pin_memory=_pin, drop_last=True,
                          persistent_workers=_persist)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=_nw, pin_memory=_pin,
                          persistent_workers=_persist)

iters_per_epoch = len(train_loader)
total_opt_steps = max(1, (N_EPOCHS * iters_per_epoch) // max(ACCUM, 1))
print(f"Train iters/epoch : {iters_per_epoch} | Total opt steps : {total_opt_steps}")

# ══════════════════════════════════════════════════════════════════════════
# 3.  DDPM SCHEDULE  (standard, same as Cell 7)
# ══════════════════════════════════════════════════════════════════════════
class DiffusionSchedule:
    def __init__(self, T=T_DIFF, b0=BETA_START, b1=BETA_END, device=DEVICE):
        betas     = torch.linspace(b0, b1, T, device=device)
        alphas    = 1.0 - betas
        alpha_bar = torch.cumprod(alphas, dim=0)
        ab_prev   = F.pad(alpha_bar[:-1], (1, 0), value=1.0)
        self.T         = T
        self.alpha_bar = alpha_bar
        self.sqrt_ab   = alpha_bar.sqrt()
        self.sqrt_1mab = (1 - alpha_bar).sqrt()
        self.post_var  = betas * (1 - ab_prev) / (1 - alpha_bar).clamp(1e-8)

    def q_sample(self, x0, t, noise=None):
        if noise is None: noise = torch.randn_like(x0)
        s  = self.sqrt_ab[t].view(-1,1,1,1)
        s1 = self.sqrt_1mab[t].view(-1,1,1,1)
        return s * x0 + s1 * noise, noise

    @torch.no_grad()
    def ddim_sample(self, model_fn, shape, ddim_steps=50, device=DEVICE):
        x   = torch.randn(shape, device=device)
        seq = list(range(self.T-1, -1, -(self.T//ddim_steps)))
        for i, ti in enumerate(seq):
            tb  = torch.full((shape[0],), ti, device=device, dtype=torch.long)
            eps = model_fn(x, tb)
            ab  = self.alpha_bar[ti]
            abp = (self.alpha_bar[seq[i+1]] if i+1<len(seq)
                   else torch.tensor(1.0, device=device))
            x0p = ((x - (1-ab).sqrt()*eps) / ab.sqrt().clamp(1e-8)).clamp(-1,1)
            x   = abp.sqrt()*x0p + (1-abp).sqrt()*eps
        return x

schedule = DiffusionSchedule()

# ══════════════════════════════════════════════════════════════════════════
# 4.  ARCHITECTURE
#
#  HistDiT uses a Diffusion Transformer (DiT) backbone conditioned on:
#    - H&E encoder features  (multi-scale CNN, same role as PST-Diff HEEncoder)
#    - SAM structural features (frozen ViT-B OR lightweight proxy CNN)
#
#  Conditioning mechanism: adaLN-Zero (from DiT paper, Peebles & Xie 2023)
#    γ, β = Linear(time_emb + sam_global)  →  LayerNorm(x) * (1+γ) + β
#  Plus cross-attention to SAM spatial features (HistDiT dual-stream).
# ══════════════════════════════════════════════════════════════════════════

def _flash_attn(q, k, v):
    """Force float16 before SDPA to ensure Flash Attention kernel activates."""
    orig = q.dtype
    out  = F.scaled_dot_product_attention(q.half(), k.half(), v.half())
    return out.to(orig)


# ── 4a. SAM Proxy (colab mode) ─────────────────────────────────────────────
class SAMProxy(nn.Module):
    """
    Lightweight CNN approximating SAM structural feature extraction.
    Extracts multi-scale edge/texture/structure features from H&E images.
    No download required — used for T4 pipeline verification only.

    Outputs:
      spatial : [B, SAM_FEAT_DIM, H//8, W//8]  for cross-attention
      global  : [B, SAM_FEAT_DIM]               for adaLN conditioning
    """
    def __init__(self, feat_dim=SAM_FEAT_DIM):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, feat_dim//4, 3, 2, 1), nn.GroupNorm(min(8,feat_dim//4), feat_dim//4), nn.GELU(),
            nn.Conv2d(feat_dim//4, feat_dim//2, 3, 2, 1), nn.GroupNorm(min(8,feat_dim//2), feat_dim//2), nn.GELU(),
            nn.Conv2d(feat_dim//2, feat_dim,   3, 2, 1), nn.GroupNorm(min(8,feat_dim),   feat_dim),   nn.GELU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        spatial = self.enc(x)              # [B, feat_dim, H//8, W//8]
        global_ = self.pool(spatial).flatten(1)  # [B, feat_dim]
        return spatial, global_


# ── 4b. SAM ViT-B Feature Extractor (a100 mode) ────────────────────────────
class SAMExtractor(nn.Module):
    """
    Wraps frozen SAM ViT-B image encoder.
    Resizes H&E from IMAGE_SIZE → 1024 (SAM native), extracts embeddings,
    then pools spatial features for DiT cross-attention.

    Outputs:
      spatial : [B, SAM_FEAT_DIM, H_s, W_s]  for cross-attention
      global  : [B, SAM_FEAT_DIM]              for adaLN conditioning
    """
    def __init__(self, image_encoder, sam_feat_dim=SAM_FEAT_DIM):
        super().__init__()
        self.encoder    = image_encoder   # frozen SAM ViT-B
        self.pool       = nn.AdaptiveAvgPool2d(1)
        # SAM outputs 256 channels; project if needed
        self.proj_spatial = (nn.Conv2d(256, sam_feat_dim, 1)
                              if sam_feat_dim != 256 else nn.Identity())
        self.proj_global  = (nn.Linear(256, sam_feat_dim)
                              if sam_feat_dim != 256 else nn.Identity())

    def forward(self, x_he):
        # x_he: [B, 3, IMAGE_SIZE, IMAGE_SIZE] in [-1,1]
        # Resize to SAM native 1024×1024
        x1024  = F.interpolate(x_he, (1024, 1024), mode="bilinear", align_corners=False)
        # SAM normalisation (from [-1,1] to SAM-space)
        x_norm = ((x1024 + 1) / 2 - SAM_PIXEL_MEAN) / SAM_PIXEL_STD
        # Extract SAM embeddings (frozen — no grad)
        with torch.no_grad():
            feats = self.encoder(x_norm)   # [B, 256, 64, 64]
        spatial = self.proj_spatial(feats)  # [B, SAM_FEAT_DIM, 64, 64]
        global_ = self.proj_global(self.pool(feats).flatten(1))  # [B, SAM_FEAT_DIM]
        return spatial, global_


# ── 4c. H&E Multi-Scale Encoder (same role as PST-Diff HEEncoder) ──────────
class HEEncoder(nn.Module):
    """
    Multi-scale H&E encoder → provides pixel-level conditioning to DiT
    via cross-attention at the patch scale.
    Identical architecture to Cell 7/8 for fair comparison.
    """
    def __init__(self, ngf=NGF):
        super().__init__()
        self.e1 = nn.Sequential(nn.Conv2d(3, ngf, 3,1,1), nn.GroupNorm(min(8,ngf), ngf), nn.SiLU())
        self.e2 = nn.Sequential(nn.Conv2d(ngf, ngf*2, 3,2,1), nn.GroupNorm(min(8,ngf*2), ngf*2), nn.SiLU())
        self.e3 = nn.Sequential(nn.Conv2d(ngf*2, ngf*4, 3,2,1), nn.GroupNorm(min(8,ngf*4), ngf*4), nn.SiLU())

    def forward(self, x):
        f1 = self.e1(x)
        f2 = self.e2(f1)
        f3 = self.e3(f2)
        return f1, f2, f3


# ── 4d. Patch Embedding / Unpatching ───────────────────────────────────────
class PatchEmbed(nn.Module):
    """
    Non-overlapping patch embedding.
    [B, 3, H, W] → [B, N, D]  where N = (H/p)²
    """
    def __init__(self, img_size, patch_size, in_ch=3, embed_dim=DIT_DIM):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches  = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, embed_dim, patch_size, patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)  # [B, N, D]


class UnPatch(nn.Module):
    """
    [B, N, D] → [B, out_ch*p², N] → [B, out_ch, H, W]
    """
    def __init__(self, img_size, patch_size, embed_dim=DIT_DIM, out_ch=3):
        super().__init__()
        self.patch_size = patch_size
        self.h_patches  = img_size // patch_size
        self.proj = nn.Linear(embed_dim, out_ch * patch_size * patch_size)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        B, N, D = x.shape
        x = self.proj(x)    # [B, N, out_ch*p²]
        p = self.patch_size; h = self.h_patches
        x = x.view(B, h, h, -1, p, p)
        x = x.permute(0, 3, 1, 4, 2, 5).contiguous()
        return x.view(B, -1, h*p, h*p)


# ── 4e. adaLN-Zero DiT Block ───────────────────────────────────────────────
class DiTBlock(nn.Module):
    """
    Single DiT block with:
      1. Self-attention on patch tokens (Flash Attention)
      2. adaLN-Zero conditioning from time + SAM global features
         γ_1,β_1 scale self-attn;  γ_2,β_2 scale MLP
         α_1,α_2 gate residuals (initialised → 0 for stable training)
      3. Cross-attention on spatial SAM features (dual-stream conditioning)
      4. Cross-attention on H&E encoder features (structure fidelity)

    Reference: Peebles & Xie (ICCV 2023) + HistDiT dual-stream extension.
    """
    def __init__(self, dim, heads, mlp_ratio=4.0, use_ckpt=False):
        super().__init__()
        self.use_ckpt = use_ckpt
        self.heads    = heads
        d_head        = dim // heads

        # Layer norms (no affine — adaLN provides scale/shift)
        self.norm1  = nn.LayerNorm(dim, elementwise_affine=False)
        self.norm2  = nn.LayerNorm(dim, elementwise_affine=False)
        self.norm_ca_sam = nn.LayerNorm(dim, elementwise_affine=False)
        self.norm_ca_he  = nn.LayerNorm(dim, elementwise_affine=False)

        # Self-attention
        self.qkv  = nn.Linear(dim, dim*3, bias=False)
        self.attn_proj = nn.Linear(dim, dim)
        nn.init.zeros_(self.attn_proj.weight); nn.init.zeros_(self.attn_proj.bias)

        # Cross-attention: SAM spatial features
        self.q_sam   = nn.Linear(dim, dim, bias=False)
        self.kv_sam  = nn.Linear(SAM_FEAT_DIM, dim*2, bias=False)
        self.ca_proj_sam = nn.Linear(dim, dim)
        nn.init.zeros_(self.ca_proj_sam.weight); nn.init.zeros_(self.ca_proj_sam.bias)

        # Cross-attention: H&E encoder features
        self.q_he   = nn.Linear(dim, dim, bias=False)
        self.kv_he  = nn.Linear(NGF*4, dim*2, bias=False)
        self.ca_proj_he  = nn.Linear(dim, dim)
        nn.init.zeros_(self.ca_proj_he.weight); nn.init.zeros_(self.ca_proj_he.bias)

        # MLP
        mlp_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, dim))
        nn.init.zeros_(self.mlp[-1].weight); nn.init.zeros_(self.mlp[-1].bias)

        # adaLN-Zero modulation: 6 params (γ1,β1,α1,γ2,β2,α2) from condition
        self.adaLN = nn.Sequential(nn.SiLU(), nn.Linear(dim, dim*6))
        nn.init.zeros_(self.adaLN[-1].weight); nn.init.zeros_(self.adaLN[-1].bias)

    def _self_attn(self, x):
        B, N, D = x.shape; h = self.heads; dh = D // h
        qkv = self.qkv(x).view(B, N, 3, h, dh).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)
        out = _flash_attn(q, k, v)
        out = out.transpose(1,2).reshape(B, N, D)
        return self.attn_proj(out)

    def _cross_attn(self, x, ctx, q_proj, kv_proj, out_proj):
        B, N, D = x.shape; h = self.heads; dh = D // h
        if ctx.dim() == 4:
            ctx = ctx.flatten(2).transpose(1, 2)  # [B, H*W, C]
        q  = q_proj(x).view(B, N, h, dh).transpose(1, 2)
        kv = kv_proj(ctx).view(B, -1, 2, h, dh).permute(2,0,3,1,4)
        k, v = kv.unbind(0)
        out = _flash_attn(q, k, v)
        out = out.transpose(1,2).reshape(B, N, D)
        return out_proj(out)

    def forward(self, x, cond, sam_spatial, he_feat):
        """
        x           : [B, N, D]  patch tokens
        cond        : [B, D]     time emb + SAM global (for adaLN)
        sam_spatial : [B, SAM_FEAT_DIM, Hs, Ws]
        he_feat     : [B, NGF*4, Hh, Wh]
        """
        # adaLN modulation parameters
        mod = self.adaLN(cond)  # [B, D*6]
        g1, b1, a1, g2, b2, a2 = mod.chunk(6, dim=-1)
        g1 = g1.unsqueeze(1); b1 = b1.unsqueeze(1)
        a1 = a1.unsqueeze(1); g2 = g2.unsqueeze(1)
        b2 = b2.unsqueeze(1); a2 = a2.unsqueeze(1)

        # 1. Self-attention (adaLN gated)
        x = x + a1 * self._self_attn(
            self.norm1(x) * (1 + g1) + b1)

        # 2. Cross-attention: SAM spatial (structure-aware)
        x = x + self._cross_attn(
            self.norm_ca_sam(x), sam_spatial,
            self.q_sam, self.kv_sam, self.ca_proj_sam)

        # 3. Cross-attention: H&E features (content fidelity)
        x = x + self._cross_attn(
            self.norm_ca_he(x), he_feat,
            self.q_he, self.kv_he, self.ca_proj_he)

        # 4. MLP (adaLN gated)
        x = x + a2 * self.mlp(
            self.norm2(x) * (1 + g2) + b2)

        return x


# ── 4f. Full HistDiT Model ──────────────────────────────────────────────────
class HistDiT(nn.Module):
    """
    HistDiT: H&E-conditioned Diffusion Transformer.

    Forward pass:
      1. Patchify noisy IHC  →  [B, N, D]
      2. Add sinusoidal position embeddings
      3. Stack DiT blocks with dual conditioning (SAM + H&E)
      4. Final LayerNorm → Unpatch → predicted noise ε [B, 3, H, W]

    Conditioning:
      cond = TimeEmb(t) + SAM_GlobalProj(sam_global)  →  adaLN for each block
      sam_spatial                                       →  cross-attn stream A
      he_feat3 (deepest HE encoder level)               →  cross-attn stream B
    """
    def __init__(self, img_size=IMAGE_SIZE, patch_size=PATCH_SIZE,
                 dim=DIT_DIM, depth=DIT_DEPTH, heads=DIT_HEADS,
                 use_ckpt=USE_GRAD_CKPT):
        super().__init__()
        self.use_ckpt  = use_ckpt
        n_patches      = (img_size // patch_size) ** 2

        # Patchify / unpatch
        self.patch_embed = PatchEmbed(img_size, patch_size, in_ch=3, embed_dim=dim)
        self.unpatch     = UnPatch(img_size, patch_size, embed_dim=dim, out_ch=3)

        # Learnable position embeddings
        self.pos_emb = nn.Parameter(torch.zeros(1, n_patches, dim))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

        # Time embedding
        self.time_emb = nn.Sequential(
            nn.Linear(dim, dim*4), nn.SiLU(),
            nn.Linear(dim*4, dim))

        # SAM global → conditioning dim
        self.sam_global_proj = nn.Sequential(
            nn.Linear(SAM_FEAT_DIM, dim), nn.SiLU(), nn.Linear(dim, dim))

        # DiT blocks
        self.blocks = nn.ModuleList([
            DiTBlock(dim, heads, use_ckpt=use_ckpt)
            for _ in range(depth)])

        # Final norm + unpatch
        self.final_norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.final_adaLN = nn.Sequential(nn.SiLU(), nn.Linear(dim, dim*2))
        nn.init.zeros_(self.final_adaLN[-1].weight)
        nn.init.zeros_(self.final_adaLN[-1].bias)

    def _time_sinusoid(self, t):
        half = DIT_DIM // 2
        freq = torch.exp(-math.log(10000) *
                         torch.arange(half, device=t.device) / max(half-1, 1))
        emb  = t.float()[:, None] * freq[None]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

    def forward(self, x_t, t, sam_spatial, sam_global, he_feats):
        """
        x_t        : [B, 3, H, W]      noisy IHC
        t          : [B]               timestep
        sam_spatial: [B, SAM_FD, Hs, Ws]
        sam_global : [B, SAM_FD]
        he_feats   : tuple (f1, f2, f3) from HEEncoder
        """
        _, _, f3 = he_feats   # use deepest HE scale for cross-attn

        # Conditioning vector: time + SAM global
        t_emb   = self.time_emb(self._time_sinusoid(t))   # [B, D]
        sam_g   = self.sam_global_proj(sam_global)          # [B, D]
        cond    = t_emb + sam_g                             # [B, D]

        # Patchify + position embed
        x = self.patch_embed(x_t) + self.pos_emb           # [B, N, D]

        # DiT blocks
        for blk in self.blocks:
            if self.use_ckpt and self.training:
                x = grad_checkpoint(blk, x, cond, sam_spatial, f3,
                                    use_reentrant=False)
            else:
                x = blk(x, cond, sam_spatial, f3)

        # Final norm with adaLN
        mod  = self.final_adaLN(cond)
        g, b = mod.chunk(2, dim=-1)
        x    = self.final_norm(x) * (1 + g.unsqueeze(1)) + b.unsqueeze(1)

        return self.unpatch(x)   # [B, 3, H, W]


# ══════════════════════════════════════════════════════════════════════════
# 5.  SAM STRUCTURAL CONSISTENCY LOSS
#
#  Penalises divergence between SAM features of predicted and real IHC.
#  Since SAM is structure-aware, this enforces "diagnostic consistency":
#  the diffusion process should alter chromogen intensity (H→IHC staining)
#  without hallucinating non-existent cellular structures.
#  Source: HistDiT (#20), SAM conditioning concept (#25).
# ══════════════════════════════════════════════════════════════════════════
def sam_structural_loss(pred_ihc, real_ihc, sam_enc):
    """
    L1 distance between SAM spatial features of predicted vs real IHC.
    Ensures structural morphology is preserved across the diffusion process.
    """
    with torch.no_grad():
        if USE_REAL_SAM:
            # Run SAM on predicted (no_grad on SAM encoder itself)
            p1024 = F.interpolate(pred_ihc.detach(), (1024,1024),
                                   mode="bilinear", align_corners=False)
            r1024 = F.interpolate(real_ihc.detach(), (1024,1024),
                                   mode="bilinear", align_corners=False)
            pn = ((p1024+1)/2 - SAM_PIXEL_MEAN) / SAM_PIXEL_STD
            rn = ((r1024+1)/2 - SAM_PIXEL_MEAN) / SAM_PIXEL_STD
            feat_p = sam_enc(pn)
            feat_r = sam_enc(rn)
        else:
            # Proxy: compare f3 features (deepest scale) as structure proxy
            feat_p = pred_ihc.detach()
            feat_r = real_ihc.detach()
    return F.l1_loss(feat_p, feat_r)


# ══════════════════════════════════════════════════════════════════════════
# 6.  MODEL ASSEMBLY
# ══════════════════════════════════════════════════════════════════════════
print(f"\nBuilding HistDiT + SAM  [{MODE.upper()} mode]...")

if USE_REAL_SAM:
    sam_enc = SAMExtractor(sam_image_encoder, SAM_FEAT_DIM).to(DEVICE)
else:
    sam_enc = SAMProxy(feat_dim=SAM_FEAT_DIM).to(DEVICE)

he_enc = HEEncoder(ngf=NGF).to(DEVICE)
dit    = HistDiT(img_size=IMAGE_SIZE, patch_size=PATCH_SIZE,
                 dim=DIT_DIM, depth=DIT_DEPTH, heads=DIT_HEADS,
                 use_ckpt=USE_GRAD_CKPT).to(DEVICE)

def cp(m): return sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6
sam_params  = cp(sam_enc)
he_params   = cp(he_enc)
dit_params  = cp(dit)
total_params= sam_params + he_params + dit_params

print(f"  SAM Enc  : {sam_params:.2f}M {'(frozen ViT-B backbone)' if USE_REAL_SAM else '(proxy CNN)'}")
print(f"  HEEncoder: {he_params:.2f}M")
print(f"  HistDiT  : {dit_params:.2f}M  "
      f"({DIT_DEPTH}×DiT blocks, adaLN-Zero + dual cross-attn)")
print(f"  Total    : {total_params:.2f}M trainable")

# Trainable params: SAM projector (if a100), HEEncoder, DiT
trainable = (list(sam_enc.parameters()) +   # includes proj layers
             list(he_enc.parameters())  +
             list(dit.parameters()))
# Note: if USE_REAL_SAM, sam_image_encoder params are frozen (requires_grad=False)

opt    = AdamW(trainable, lr=LR, betas=(0.9, 0.999), weight_decay=1e-4)
sched  = CosineAnnealingLR(opt, T_max=total_opt_steps, eta_min=LR*0.01)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# ══════════════════════════════════════════════════════════════════════════
# 7.  METRICS
# ══════════════════════════════════════════════════════════════════════════
def psnr(a, b):
    return 10*math.log10(4.0/(((a.clamp(-1,1)-b.clamp(-1,1))**2).mean().item()+1e-8))

def ssim_fast(a, b):
    k, p = 11, 5
    ma, mb = F.avg_pool2d(a,k,1,p), F.avg_pool2d(b,k,1,p)
    sa = F.avg_pool2d(a**2,k,1,p)-ma**2; sb = F.avg_pool2d(b**2,k,1,p)-mb**2
    sab= F.avg_pool2d(a*b, k,1,p)-ma*mb; C1, C2 = 0.01**2, 0.03**2
    return (((2*ma*mb+C1)*(2*sab+C2))/((ma**2+mb**2+C1)*(sa+sb+C2)).clamp_min(1e-8)).mean().item()

# ══════════════════════════════════════════════════════════════════════════
# 8.  CSV HELPERS  (same column names as Cells 7 & 8 for direct comparison)
# ══════════════════════════════════════════════════════════════════════════
def init_csv(p, f):
    with open(p,"w",newline="",buffering=1) as fp: csv.writer(fp).writerow(f)

def append_csv(p, r):
    with open(p,"a",newline="",buffering=1) as fp: csv.writer(fp).writerow(r)

init_csv(LOG_CSV,   ["wall_time","epoch","iter","global_step",
                     "loss_total","loss_diffusion","loss_sam"])
init_csv(EPOCH_CSV, ["wall_time","epoch","mean_loss_total",
                     "mean_loss_diffusion","mean_loss_sam",
                     "val_psnr","val_ssim"])

# ══════════════════════════════════════════════════════════════════════════
# 9.  SAMPLE SAVER  (same recon/target convention as evaluate.py)
# ══════════════════════════════════════════════════════════════════════════
def save_samples(epoch, he, x0_pred, real_ihc, global_step=0, tag="epoch"):
    def norm(t): return ((t.detach().cpu().clamp(-1,1)+1)/2)
    imgs  = torch.cat([norm(he), norm(x0_pred), norm(real_ihc)], dim=0)
    grid  = vutils.make_grid(imgs, nrow=he.shape[0], padding=2, normalize=False)
    fname = os.path.join(SAMPLE_DIR, f"{tag}_ep{epoch:04d}_step{global_step:07d}.png")
    vutils.save_image(grid, fname)
    print(f"  [Sample] {fname}")

# ══════════════════════════════════════════════════════════════════════════
# 10.  VALIDATION
# ══════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def validate(epoch, max_batches=MAX_VAL_BATCHES, ddim_steps=VAL_DDIM_STEPS):
    sam_enc.eval(); he_enc.eval(); dit.eval()
    psnr_s, ssim_s = [], []

    for i, (he, ihc) in enumerate(val_loader):
        if i >= max_batches: break
        he, ihc = he.to(DEVICE), ihc.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            sam_sp, sam_gl = sam_enc(he)
            he_feats        = he_enc(he)

            def model_fn(x, t_b):
                return dit(x, t_b, sam_sp, sam_gl, he_feats)

            fake = schedule.ddim_sample(model_fn, he.shape,
                                         ddim_steps=ddim_steps, device=DEVICE)

        psnr_s.append(psnr(fake, ihc))
        ssim_s.append(ssim_fast(fake, ihc))
        torch.cuda.empty_cache()

    sam_enc.train(); he_enc.train(); dit.train()
    return float(np.mean(psnr_s)), float(np.mean(ssim_s))

# ══════════════════════════════════════════════════════════════════════════
# 11.  CHECKPOINT HELPER
# ══════════════════════════════════════════════════════════════════════════
def save_ckpt(path, epoch, opt_step, val_psnr, val_ssim):
    torch.save({
        "epoch": epoch, "opt_step": opt_step, "mode": MODE,
        "sam_enc":  sam_enc.state_dict(),
        "he_enc":   he_enc.state_dict(),
        "dit":      dit.state_dict(),
        "opt":      opt.state_dict(),
        "sched":    sched.state_dict(),
        "scaler":   scaler.state_dict(),
        "val_psnr": val_psnr, "val_ssim": val_ssim,
        "config": {
            "IMAGE_SIZE":IMAGE_SIZE,"PATCH_SIZE":PATCH_SIZE,
            "DIT_DIM":DIT_DIM,"DIT_DEPTH":DIT_DEPTH,"DIT_HEADS":DIT_HEADS,
            "NGF":NGF,"T_DIFF":T_DIFF,"MODE":MODE,"USE_REAL_SAM":USE_REAL_SAM}
    }, path)
    print(f"  [Ckpt] → {path}")

def backup(ckpt):
    try:
        shutil.copy2(ckpt, os.path.join(DRIVE_CKPT, os.path.basename(ckpt)))
        print(f"  [Backup] saved")
    except Exception as e:
        print(f"  [Backup] failed (non-fatal): {e}")

# ══════════════════════════════════════════════════════════════════════════
# 12.  TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"  HistDiT + SAM  [{MODE.upper()}]")
print(f"  {IMAGE_SIZE}px | Patch={PATCH_SIZE} | DiT {DIT_DIM}d×{DIT_DEPTH}L | "
      f"SAM={'ViT-B' if USE_REAL_SAM else 'Proxy'}")
print(f"  Epochs={N_EPOCHS} | EffBatch={BATCH_SIZE*ACCUM} | T={T_DIFF}")
print(f"  Loss: L2(noise) + SAM structural consistency")
print(f"  Ckpt at epochs {EXPLICIT_CKPT_EPOCHS} + latest on timeout")
print("="*60 + "\n")

global_step  = 0
opt_step     = 0
wall_start   = time.time()
timeout_flag = False
last_val_psnr= 0.0
last_val_ssim= 0.0
last_epoch   = 0

for epoch in range(1, N_EPOCHS+1):
    if timeout_flag: break

    ep = {"total":[], "diff":[], "sam":[]}
    opt.zero_grad()

    _vh, _vi = next(iter(val_loader))
    _vh, _vi = _vh.to(DEVICE), _vi.to(DEVICE)

    for it, (real_he, real_ihc) in enumerate(train_loader):
        global_step += 1
        real_he  = real_he.to(DEVICE, non_blocking=True)
        real_ihc = real_ihc.to(DEVICE, non_blocking=True)
        t        = torch.randint(0, T_DIFF, (real_he.shape[0],), device=DEVICE)

        # ── 1. SAM feature extraction ──────────────────────────────────
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            sam_spatial, sam_global = sam_enc(real_he)

        # ── 2. H&E encode ──────────────────────────────────────────────
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            he_feats = he_enc(real_he)

        # ── 3. Perturb IHC + DiT forward ───────────────────────────────
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            x_t, eps_true = schedule.q_sample(real_ihc, t)

            # DiT predicts noise ε (L2 loss — DiT standard)
            eps_pred = dit(x_t, t, sam_spatial, sam_global, he_feats)
            l_diff   = F.mse_loss(eps_pred, eps_true)

            # Recover x0 estimate for SAM structural loss
            ab      = schedule.alpha_bar[t].view(-1,1,1,1)
            x0_pred = ((x_t - (1-ab).sqrt()*eps_pred) / ab.sqrt().clamp(1e-8)).clamp(-1,1)

            # ── 4. SAM structural consistency loss ──────────────────────
            l_sam = sam_structural_loss(x0_pred, real_ihc,
                                         sam_image_encoder if USE_REAL_SAM else None)

            loss = l_diff + LAMBDA_SAM * l_sam

        # ── 5. Backward ─────────────────────────────────────────────────
        scaler.scale(loss / max(ACCUM, 1)).backward()

        if global_step % max(ACCUM, 1) == 0:
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update()
            opt.zero_grad(); sched.step()
            opt_step += 1

        ep["total"].append(loss.item())
        ep["diff"].append(l_diff.item())
        ep["sam"].append(l_sam.item())

        # ── Logging ─────────────────────────────────────────────────────
        if global_step % SAVE_EVERY == 0:
            ts  = datetime.now().isoformat(timespec="seconds")
            elm = (time.time()-wall_start)/60
            append_csv(LOG_CSV, [ts, epoch, it+1, global_step,
                                  f"{loss.item():.6f}",
                                  f"{l_diff.item():.6f}",
                                  f"{l_sam.item():.6f}"])
            print(f"[{ts}] Ep{epoch}/{N_EPOCHS} It{it+1} {elm:.1f}m | "
                  f"loss={loss.item():.4f} diff={l_diff.item():.4f} "
                  f"sam={l_sam.item():.4f}")

        if global_step % SAMPLE_EVERY == 0:
            save_samples(epoch, real_he, x0_pred, real_ihc,
                         global_step=global_step, tag="iter")

        if time.time()-wall_start > MAX_WALL_SECS:
            print(f"\n[TIMEOUT] Wall cap reached at epoch {epoch} iter {it+1}.")
            timeout_flag = True; break

    # ── Epoch end ─────────────────────────────────────────────────────────
    last_epoch = epoch
    ml = {k: float(np.mean(v)) for k, v in ep.items()}
    torch.cuda.empty_cache(); gc.collect()

    # Only save / validate / checkpoint every CKPT_EVERY_N epochs
    if epoch % CKPT_EVERY_N == 0 or epoch in EXPLICIT_CKPT_EPOCHS or timeout_flag:
        val_psnr, val_ssim = validate(epoch)
        last_val_psnr = val_psnr; last_val_ssim = val_ssim

        ts = datetime.now().isoformat(timespec="seconds")
        append_csv(EPOCH_CSV, [ts, epoch,
                                f"{ml['total']:.6f}", f"{ml['diff']:.6f}",
                                f"{ml['sam']:.6f}", f"{val_psnr:.4f}",
                                f"{val_ssim:.6f}"])

        ckpt = os.path.join(CKPT_DIR, f"epoch_{epoch:04d}.pt")
        save_ckpt(ckpt, epoch, opt_step, val_psnr, val_ssim)

        if epoch in EXPLICIT_CKPT_EPOCHS:
            explicit = os.path.join(CKPT_DIR, f"epoch_{epoch:04d}_explicit.pt")
            shutil.copy2(ckpt, explicit)
            print(f"  [Ckpt] Explicit epoch {epoch} → {explicit}")

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                _sp, _sg = sam_enc(_vh)
                _hf      = he_enc(_vh)
                def _mfn(x, tb): return dit(x, tb, _sp, _sg, _hf)
                _s = schedule.ddim_sample(_mfn, _vi.shape,
                                           ddim_steps=VAL_DDIM_STEPS, device=DEVICE)
        save_samples(epoch, _vh, _s, _vi, global_step=global_step, tag="epoch")
        backup(ckpt)

    elapsed = (time.time()-wall_start)/60
    print(f"\n{'='*60}")
    print(f"  Epoch {epoch}/{N_EPOCHS} [{MODE.upper()}] | {elapsed:.1f}min")
    print(f"  Loss  total={ml['total']:.4f}  diff={ml['diff']:.4f}  sam={ml['sam']:.4f}")
    if epoch % CKPT_EVERY_N == 0 or epoch in EXPLICIT_CKPT_EPOCHS:
        print(f"  Val   PSNR={val_psnr:.4f} dB  SSIM={val_ssim:.6f}")
    print(f"{'='*60}\n")

    if timeout_flag: break

# ── Final timeout checkpoint ─────────────────────────────────────────────
if timeout_flag and last_epoch > 0:
    final = os.path.join(CKPT_DIR, f"final_epoch{last_epoch}.pt")
    save_ckpt(final, last_epoch, opt_step, last_val_psnr, last_val_ssim)
    backup(final)
    print(f"  [Ckpt] Final checkpoint → {final}")

print("\nTraining complete.")
print(f"  Mode     : {MODE.upper()}")
print(f"  Iter CSV : {LOG_CSV}")
print(f"  Epoch CSV: {EPOCH_CSV}")
print(f"  Ckpts    : {CKPT_DIR}")
print(f"  Samples  : {SAMPLE_DIR}")

GPU memory before build: 0.00 GiB  (should be near 0)
Device : cuda | GPU : NVIDIA A100-SXM4-40GB
Mode   : A100 | 512px | Patch=16 | DiT=512d×12L×8h | SAM=Proxy

Loading datasets...
  [train] 3311 pairs
  [val  ] 585 pairs
Train iters/epoch : 206 | Total opt steps : 5150

Building HistDiT + SAM  [A100 mode]...
  SAM Enc  : 0.37M (proxy CNN)
  HEEncoder: 0.37M
  HistDiT  : 79.91M  (12×DiT blocks, adaLN-Zero + dual cross-attn)
  Total    : 80.66M trainable

  HistDiT + SAM  [A100]
  512px | Patch=16 | DiT 512d×12L | SAM=Proxy
  Epochs=50 | EffBatch=32 | T=1000
  Loss: L2(noise) + SAM structural consistency
  Ckpt at epochs {8} + latest on timeout

[2026-05-07T05:27:41] Ep1/50 It100 0.8m | loss=0.6522 diff=0.5878 sam=0.6448


KeyboardInterrupt: 

Cell 11: Checkpoint Recovery

In [16]:
# ══════════════════════════════════════════════════════════════
#  CHECKPOINT RECOVERY — rebuild all 4 metrics from Drive
#  Run this after mounting Drive in a fresh session
# ══════════════════════════════════════════════════════════════
import os, gc, math, torch, numpy as np
import torch.nn.functional as F
import torchvision.transforms as T
from pathlib import Path
from PIL import Image
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Verify what checkpoints exist on Drive
DRIVE_DIRS = {
    "BBDM":         "/content/drive/MyDrive/BBDM_checkpoints_a100",
    "PST-Diff+STN": "/content/drive/MyDrive/PST_STN_checkpoints_a100",
    "ScoreTopo":    "/content/drive/MyDrive/ScoreTopo_checkpoints_a100",
    "HistDiT+SAM":  "/content/drive/MyDrive/HistDiT_SAM_checkpoints_a100",
}

print("Checkpoints found on Drive:")
for name, d in DRIVE_DIRS.items():
    if os.path.exists(d):
        ckpts = sorted(Path(d).glob("*.pt"))
        print(f"  {name}: {len(ckpts)} files → {[f.name for f in ckpts[-3:]]}")
    else:
        print(f"  {name}: ✗ directory not found at {d}")

Checkpoints found on Drive:
  BBDM: 0 files → []
  PST-Diff+STN: 6 files → ['epoch_0030.pt', 'epoch_0035.pt', 'final_epoch35.pt']
  ScoreTopo: 6 files → ['epoch_0030.pt', 'epoch_0040.pt', 'final_epoch40.pt']
  HistDiT+SAM: 11 files → ['epoch_0030.pt', 'epoch_0040.pt', 'epoch_0050.pt']


Cell 12: Complete Analysis of All 4 Models (run after all computation is complete)

In [11]:
# ══════════════════════════════════════════════════════════════
#  FULL EVALUATION — all 4 models, all 4 metrics
#  Prereqs: Drive mounted, dataset extracted, model cells
#  re-run (to define class architectures) but NOT retrained
# ══════════════════════════════════════════════════════════════

import os, gc, math, shutil, torch, numpy as np, subprocess, sys
import torch.nn.functional as F
import torchvision.transforms as T
from pathlib import Path
from PIL import Image as PILImage
from torch.utils.data import DataLoader, Dataset

try: import lpips
except: os.system("pip install -q lpips"); import lpips
try: from pytorch_fid import fid_score
except: os.system("pip install -q pytorch-fid"); from pytorch_fid import fid_score

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = "/content/datasets_standardized/BCI_512"
OUT_DIR   = "/content/results/analysis"
os.makedirs(OUT_DIR, exist_ok=True)

loss_fn = lpips.LPIPS(net='alex', verbose=False).to(DEVICE)
to_t    = lambda p: T.ToTensor()(PILImage.open(p).convert('RGB'))*2-1

def _psnr(a,b):
    return 10*math.log10(4.0/(((a.clamp(-1,1)-b.clamp(-1,1))**2).mean().item()+1e-8))
def _ssim(a,b):
    k,p=11,5; ma,mb=F.avg_pool2d(a,k,1,p),F.avg_pool2d(b,k,1,p)
    sa=F.avg_pool2d(a**2,k,1,p)-ma**2; sb=F.avg_pool2d(b**2,k,1,p)-mb**2
    sab=F.avg_pool2d(a*b,k,1,p)-ma*mb; C1,C2=0.01**2,0.03**2
    return (((2*ma*mb+C1)*(2*sab+C2))/((ma**2+mb**2+C1)*(sa+sb+C2)).clamp_min(1e-8)).mean().item()

def compute_metrics(recon_dir, target_dir):
    r_files = sorted(Path(recon_dir).glob("*.png"))
    t_files = sorted(Path(target_dir).glob("*.png"))
    ps,ss,ls = [],[],[]
    for rf,tf in zip(r_files, t_files):
        p = to_t(rf).unsqueeze(0).to(DEVICE)
        g = to_t(tf).unsqueeze(0).to(DEVICE)
        ps.append(_psnr(p,g)); ss.append(_ssim(p,g))
        with torch.no_grad(): ls.append(loss_fn(p,g).item())
    fid_v = fid_score.calculate_fid_given_paths(
        [str(recon_dir), str(target_dir)],
        batch_size=16, device=str(DEVICE), dims=2048)
    return np.mean(ps), np.mean(ss), np.mean(ls), fid_v

# Simple val dataset
class ValDS(Dataset):
    def __init__(self, size=256):
        d = os.path.join(DATA_ROOT,"val")
        self.pairs = sorted([
            (os.path.join(d,"A",f), os.path.join(d,"B",f.replace("_A.","_B.")))
            for f in os.listdir(os.path.join(d,"A")) if f.endswith(".png")
            and os.path.exists(os.path.join(d,"B",f.replace("_A.","_B.")))])
        self.size = size
    def __len__(self): return len(self.pairs)
    def _t(self,p):
        return T.ToTensor()(PILImage.open(p).convert("RGB")
               .resize((self.size,self.size),PILImage.BICUBIC))*2-1
    def __getitem__(self,i):
        a,b=self.pairs[i]; return self._t(a), self._t(b)

all_results = {}

# ══════════════════════════════════════════════════════════════
#  MODEL 1: BBDM — run --sample_to_eval subprocess (~20 min)
# ══════════════════════════════════════════════════════════════
# ── BBDM skipped — results already known ──────────────────────
# print("="*60)
# print("  BBDM — running sample_to_eval from checkpoint...")
# print("="*60)

# os.chdir('/content/BBDM')
# bbdm_ckpt    = "/content/BBDM/results/BCI_512/BBDM_A100/checkpoint/last_model.pth"
# bbdm_eval    = Path("/content/BBDM/results/BCI_512/BBDM_A100/sample_to_eval")
# bbdm_recon   = Path(OUT_DIR)/"bbdm_recon";  bbdm_recon.mkdir(exist_ok=True)
# bbdm_target  = Path(OUT_DIR)/"bbdm_target"; bbdm_target.mkdir(exist_ok=True)
# real_ihcs    = sorted(Path(DATA_ROOT,"val","B").glob("*.png"))

# import yaml
# with open("configs/My_Method_config.yaml") as f:
#     cfg = yaml.safe_load(f)
# cfg['model']['BB']['params']['sample_step'] = 50
# cfg['testing']['sample_num'] = 1
# with open("configs/My_Method_config.yaml","w") as f:
#     yaml.dump(cfg, f, default_flow_style=False)
# print("  Config patched: sample_step=50")

# result = subprocess.run([
#     "python3", "main.py",
#     "--config", "configs/My_Method_config.yaml",
#     "--sample_to_eval", "--gpu_ids", "0",
#     "--resume_model", bbdm_ckpt
# ], capture_output=True, text=True, timeout=7200)

# print(result.stdout[-1000:] if result.stdout else "")
# if result.returncode != 0:
#     print(f"  stderr: {result.stderr[-500:]}")

# bbdm_imgs = sorted((bbdm_eval / "50").glob("*.png"))
# bbdm_gts  = sorted((bbdm_eval / "ground_truth").glob("*.png"))
# print(f"  Generated {len(bbdm_imgs)} images")

# if bbdm_imgs:
#     for i,(gf,tf) in enumerate(zip(bbdm_imgs, bbdm_gts)):
#         img = PILImage.open(gf).convert("RGB")
#         img.resize((256,256),PILImage.BICUBIC).save(bbdm_recon/f"{i:05d}.png")
#         PILImage.open(tf).convert("RGB").resize((256,256),PILImage.BICUBIC).save(bbdm_target/f"{i:05d}.png")
#     p,s,l,f = compute_metrics(bbdm_recon, bbdm_target)
#     all_results["BBDM"] = dict(psnr=p,ssim=s,lpips=l,fid=f,epoch=47,n=len(bbdm_imgs))
#     print(f"  PSNR={p:.4f}  SSIM={s:.6f}  LPIPS={l:.4f}  FID={f:.2f}")
# else:
#     print("  ✗ BBDM sampling produced no images")

# gc.collect(); torch.cuda.empty_cache()

# Hardcoded from previous run
all_results["BBDM"] = dict(psnr=15.1564, ssim=0.242925, lpips=0.7570, fid=360.96, epoch=47, n=977)

# ══════════════════════════════════════════════════════════════
#  MODEL 2: PST-Diff+STN — load epoch 35 from Drive
#  Requires: Cell 7 (PST-Diff) already re-run to define classes
# ══════════════════════════════════════════════════════════════
# ── PST-Diff+STN skipped — results already known ──────────────
# print("\n"+"="*60)
# print("  PST-Diff+STN — loading epoch 35 from Drive...")
# print("="*60)

# try:
#     ckpt = torch.load(
#         "/content/drive/MyDrive/PST_STN_checkpoints_a100/final_epoch35.pt",
#         map_location=DEVICE)
#     he_enc.load_state_dict(ckpt["he_enc"]); he_enc.eval()
#     unet.load_state_dict(ckpt["unet"]);     unet.eval()
#     if ckpt.get("stn") and stn is not None:
#         stn.load_state_dict(ckpt["stn"]);   stn.eval()

#     pst_recon  = Path(OUT_DIR)/"pst_recon";  pst_recon.mkdir(exist_ok=True)
#     pst_target = Path(OUT_DIR)/"pst_target"; pst_target.mkdir(exist_ok=True)
#     val_loader = DataLoader(ValDS(256), batch_size=1, shuffle=False, num_workers=2)

#     with torch.no_grad():
#         for i,(he,ihc) in enumerate(val_loader):
#             he,ihc = he.to(DEVICE),ihc.to(DEVICE)
#             with torch.amp.autocast('cuda',dtype=torch.bfloat16):
#                 feats = he_enc(he)
#                 fake  = schedule.ddim_sample(
#                     lambda x,t,_: unet(x,t,feats),feats,he.shape,ddim_steps=50)
#             fake = fake.nan_to_num(0.0).clamp(-1,1)
#             PILImage.fromarray(
#                 ((fake[0].cpu()+1)/2*255).permute(1,2,0).byte().numpy()
#             ).save(pst_recon/f"{i:05d}.png")
#             PILImage.fromarray(
#                 ((ihc[0].cpu()+1)/2*255).permute(1,2,0).byte().numpy()
#             ).save(pst_target/f"{i:05d}.png")
#             if i%50==0: print(f"  [{i}/585]")

#     p,s,l,f = compute_metrics(pst_recon, pst_target)
#     all_results["PST-Diff+STN"] = dict(psnr=p,ssim=s,lpips=l,fid=f,epoch=35,n=585)
#     print(f"  PSNR={p:.4f}  SSIM={s:.6f}  LPIPS={l:.4f}  FID={f:.2f}")
# except Exception as e:
#     print(f"  ✗ PST-Diff+STN failed: {e}")
#     print("  → Re-run Cell 7 (PST-Diff, MODE='a100') first to define classes")

# gc.collect(); torch.cuda.empty_cache()

# Hardcoded from previous run
all_results["PST-Diff+STN"] = dict(psnr=10.0750, ssim=0.015514, lpips=0.6643, fid=214.77, epoch=35, n=585)

# ══════════════════════════════════════════════════════════════
#  MODEL 3: ScoreTopo — skipped, too slow
# ══════════════════════════════════════════════════════════════
# try:
#     ckpt = torch.load(
#         "/content/drive/MyDrive/ScoreTopo_checkpoints_a100/final_epoch40.pt",
#         map_location=DEVICE)
#     he_enc.load_state_dict(ckpt["he_enc"]);         he_enc.eval()
#     score_unet.load_state_dict(ckpt["score_unet"]); score_unet.eval()

#     sc_recon  = Path(OUT_DIR)/"score_recon";  sc_recon.mkdir(exist_ok=True)
#     sc_target = Path(OUT_DIR)/"score_target"; sc_target.mkdir(exist_ok=True)
#     val_loader = DataLoader(ValDS(256), batch_size=1, shuffle=False, num_workers=2)

#     with torch.no_grad():
#         for i,(he,ihc) in enumerate(val_loader):
#             if i >= 50: break
#             he,ihc = he.to(DEVICE),ihc.to(DEVICE)
#             print(f"  processing [{i}/585]...", flush=True)
#             with torch.amp.autocast('cuda',dtype=torch.bfloat16):
#                 feats = he_enc(he)
#                 fake  = schedule.sample(
#                     lambda x,t: score_unet(x,t,feats),
#                     he.shape, device=DEVICE)
#             fake = fake.nan_to_num(0.0).clamp(-1,1)
#             PILImage.fromarray(
#                 ((fake[0].cpu()+1)/2*255).permute(1,2,0).byte().numpy()
#             ).save(sc_recon/f"{i:05d}.png")
#             PILImage.fromarray(
#                 ((ihc[0].cpu()+1)/2*255).permute(1,2,0).byte().numpy()
#             ).save(sc_target/f"{i:05d}.png")
#             if i%50==0: print(f"  [{i}/585]")

#     p,s,l,f = compute_metrics(sc_recon, sc_target)
#     all_results["ScoreTopo"] = dict(psnr=p,ssim=s,lpips=l,fid=f,epoch=40,n=585)
#     print(f"  PSNR={p:.4f}  SSIM={s:.6f}  LPIPS={l:.4f}  FID={f:.2f}")
# except Exception as e:
#     print(f"  ✗ ScoreTopo failed: {e}")
#     print("  → Re-run ScoreTopo cell (MODE='a100') first to define classes")

# gc.collect(); torch.cuda.empty_cache()

all_results["ScoreTopo"] = dict(psnr=6.8935, ssim=0.001263, lpips=1.1689, fid=416.81, epoch=40, n=585)

# ══════════════════════════════════════════════════════════════
#  MODEL 4: HistDiT+SAM — load epoch 50 from Drive
#  Requires: HistDiT cell already re-run to define classes
# ══════════════════════════════════════════════════════════════
he_enc = HEEncoder(64).to(DEVICE)  # rebuild for HistDiT NGF
print("\n"+"="*60)
print("  HistDiT+SAM — loading epoch 50 from Drive...")
print("="*60)

try:
    if not hasattr(dit, '_eval_loaded'):
        ckpt = torch.load(
            "/content/drive/MyDrive/HistDiT_SAM_checkpoints_a100/epoch_0050.pt",
            map_location=DEVICE)
        sam_enc.load_state_dict(ckpt["sam_enc"]); sam_enc.eval()
        he_enc.load_state_dict(ckpt["he_enc"]);   he_enc.eval()
        dit.load_state_dict(ckpt["dit"]);         dit.eval()
        dit._eval_loaded = True
        print("  Loaded checkpoint from Drive")
    else:
        print("  Checkpoint already loaded — skipping Drive read")

    hd_recon  = Path(OUT_DIR)/"histdit_recon";  hd_recon.mkdir(exist_ok=True)
    hd_target = Path(OUT_DIR)/"histdit_target"; hd_target.mkdir(exist_ok=True)
    val_loader = DataLoader(ValDS(512), batch_size=1, shuffle=False, num_workers=2)

    import time
    t0 = time.time()
    with torch.no_grad():
        for i,(he,ihc) in enumerate(val_loader):
            he,ihc = he.to(DEVICE),ihc.to(DEVICE)
            if i > 0:
                elapsed = time.time()-t0
                eta = elapsed/i*(585-i)
                print(f"  [{i}/585] {elapsed/60:.1f}min elapsed, ETA {eta/60:.1f}min", flush=True)

            with torch.cuda.amp.autocast(enabled=True):
                sam_sp, sam_gl = sam_enc(he)
                he_feats = he_enc(he)
                def _model_fn(x, t_b):
                    return dit(x, t_b, sam_sp, sam_gl, he_feats)
                fake = schedule.ddim_sample(_model_fn, he.shape, ddim_steps=50, device=DEVICE)
            fake = fake.nan_to_num(0.0).clamp(-1,1)
            PILImage.fromarray(
                ((fake[0].cpu()+1)/2*255).permute(1,2,0).byte().numpy()
            ).resize((256,256), PILImage.BICUBIC).save(hd_recon/f"{i:05d}.png")
            PILImage.fromarray(
                ((ihc[0].cpu()+1)/2*255).permute(1,2,0).byte().numpy()
            ).save(hd_target/f"{i:05d}.png")
            if i%50==0: print(f"  [{i}/585]")

    p,s,l,f = compute_metrics(hd_recon, hd_target)
    all_results["HistDiT+SAM"] = dict(psnr=p,ssim=s,lpips=l,fid=f,epoch=50,n=585)
    print(f"  PSNR={p:.4f}  SSIM={s:.6f}  LPIPS={l:.4f}  FID={f:.2f}")
except Exception as e:
    import traceback
    print(f"  ✗ HistDiT+SAM failed: {e}")
    traceback.print_exc()

gc.collect(); torch.cuda.empty_cache()

all_results["PST-Diff+STN"] = dict(psnr=10.0750, ssim=0.015514, lpips=0.6643, fid=214.77, epoch=35, n=585)

all_results["PST-Diff+STN"] = dict(psnr=10.0750, ssim=0.015514, lpips=0.6643, fid=214.77, epoch=35, n=585)

# ══════════════════════════════════════════════════════════════
#  FINAL SUMMARY
# ══════════════════════════════════════════════════════════════
print("\n"+"="*60)
print("  FINAL RESULTS — all 4 models")
print("="*60)
print(f"{'Model':<18} {'Ep':<5} {'N':<6} {'PSNR':<10} {'SSIM':<12} {'LPIPS':<10} {'FID'}")
print("-"*65)
for name,m in all_results.items():
    print(f"{name:<18} {m['epoch']:<5} {m['n']:<6} "
          f"{m['psnr']:.4f}{'':4} {m['ssim']:.6f}{'':4} "
          f"{m['lpips']:.4f}{'':4} {m['fid']:.2f}")

print("\nLaTeX table:")
print("\\begin{tabular}{lcccc}\\toprule")
print("Model & PSNR$\\uparrow$ & SSIM$\\uparrow$ & LPIPS$\\downarrow$ & FID$\\downarrow$ \\\\\\midrule")
for name,m in all_results.items():
    print(f"{name} & {m['psnr']:.4f} & {m['ssim']:.6f} & {m['lpips']:.4f} & {m['fid']:.2f} \\\\")
print("\\bottomrule\\end{tabular}")


  HistDiT+SAM — loading epoch 50 from Drive...
  Loaded checkpoint from Drive
  [0/585]
  [1/585] 0.0min elapsed, ETA 12.6min
  [2/585] 0.0min elapsed, ETA 12.1min
  [3/585] 0.1min elapsed, ETA 11.8min
  [4/585] 0.1min elapsed, ETA 11.6min
  [5/585] 0.1min elapsed, ETA 11.6min
  [6/585] 0.1min elapsed, ETA 11.5min
  [7/585] 0.1min elapsed, ETA 11.4min
  [8/585] 0.2min elapsed, ETA 11.4min
  [9/585] 0.2min elapsed, ETA 11.4min
  [10/585] 0.2min elapsed, ETA 11.3min
  [11/585] 0.2min elapsed, ETA 11.3min
  [12/585] 0.2min elapsed, ETA 11.3min
  [13/585] 0.3min elapsed, ETA 11.3min
  [14/585] 0.3min elapsed, ETA 11.2min
  [15/585] 0.3min elapsed, ETA 11.2min
  [16/585] 0.3min elapsed, ETA 11.2min
  [17/585] 0.3min elapsed, ETA 11.2min
  [18/585] 0.4min elapsed, ETA 11.2min
  [19/585] 0.4min elapsed, ETA 11.2min
  [20/585] 0.4min elapsed, ETA 11.1min
  [21/585] 0.4min elapsed, ETA 11.1min
  [22/585] 0.4min elapsed, ETA 11.1min
  [23/585] 0.5min elapsed, ETA 11.1min
  [24/585] 0.5min elaps

Traceback (most recent call last):
  File "/tmp/ipykernel_473/375270050.py", line 267, in <cell line: 0>
    p,s,l,f = compute_metrics(hd_recon, hd_target)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_473/375270050.py", line 42, in compute_metrics
    ps.append(_psnr(p,g)); ss.append(_ssim(p,g))
              ^^^^^^^^^^
  File "/tmp/ipykernel_473/375270050.py", line 28, in _psnr
    return 10*math.log10(4.0/(((a.clamp(-1,1)-b.clamp(-1,1))**2).mean().item()+1e-8))
                                ~~~~~~~~~~~~~^~~~~~~~~~~~~~
RuntimeError: The size of tensor a (256) must match the size of tensor b (512) at non-singleton dimension 3


In [19]:
print("\n"+"="*60)
print("  FINAL RESULTS — all 4 models")
print("="*60)
print(f"{'Model':<18} {'Ep':<5} {'N':<6} {'PSNR':<10} {'SSIM':<12} {'LPIPS':<10} {'FID'}")
print("-"*65)
for name,m in all_results.items():
    print(f"{name:<18} {m['epoch']:<5} {m['n']:<6} "
          f"{m['psnr']:.4f}{'':4} {m['ssim']:.6f}{'':4} "
          f"{m['lpips']:.4f}{'':4} {m['fid']:.2f}")

print("\nLaTeX table:")
print("\\begin{tabular}{lcccc}\\toprule")
print("Model & PSNR$\\uparrow$ & SSIM$\\uparrow$ & LPIPS$\\downarrow$ & FID$\\downarrow$ \\\\\\midrule")
for name,m in all_results.items():
    print(f"{name} & {m['psnr']:.4f} & {m['ssim']:.6f} & {m['lpips']:.4f} & {m['fid']:.2f} \\\\")
print("\\bottomrule\\end{tabular}")


  FINAL RESULTS — all 4 models
Model              Ep    N      PSNR       SSIM         LPIPS      FID
-----------------------------------------------------------------
BBDM               47    977    15.1564     0.242925     0.7570     360.96
PST-Diff+STN       35    585    10.0750     0.015514     0.6643     214.77
ScoreTopo          40    585    6.8935     0.001263     1.1689     416.81
HistDiT+SAM        50    585    11.9324     0.000409     0.9403     443.43

LaTeX table:
\begin{tabular}{lcccc}\toprule
Model & PSNR$\uparrow$ & SSIM$\uparrow$ & LPIPS$\downarrow$ & FID$\downarrow$ \\\midrule
BBDM & 15.1564 & 0.242925 & 0.7570 & 360.96 \\
PST-Diff+STN & 10.0750 & 0.015514 & 0.6643 & 214.77 \\
ScoreTopo & 6.8935 & 0.001263 & 1.1689 & 416.81 \\
HistDiT+SAM & 11.9324 & 0.000409 & 0.9403 & 443.43 \\
\bottomrule\end{tabular}


In [12]:
from PIL import Image as PILImage
from pathlib import Path

hd_recon = Path("/content/results/analysis/histdit_recon")
files = sorted(hd_recon.glob("*.png"))
print(f"Total files: {len(files)}")
print(f"First image size: {PILImage.open(files[0]).size}")
print(f"Last image size: {PILImage.open(files[-1]).size}")

Total files: 585
First image size: (256, 256)
Last image size: (256, 256)


In [16]:
from PIL import Image as PILImage
from pathlib import Path

hd_target = Path("/content/results/analysis/histdit_target")
fixed = 0
for f in sorted(hd_target.glob("*.png")):
    img = PILImage.open(f)
    print(f"Before: {f.name} = {img.size}")
    resized = img.resize((256, 256), PILImage.BICUBIC)
    resized.save(f)
    verify = PILImage.open(f)
    print(f"After:  {f.name} = {verify.size}")
    fixed += 1
    if fixed >= 3:
        break
print(f"Done — check sizes above")

Before: 00000.png = (512, 512)
After:  00000.png = (256, 256)
Before: 00001.png = (512, 512)
After:  00001.png = (256, 256)
Before: 00002.png = (512, 512)
After:  00002.png = (256, 256)
Done — check sizes above


In [17]:
from PIL import Image as PILImage
from pathlib import Path

hd_target = Path("/content/results/analysis/histdit_target")
for f in sorted(hd_target.glob("*.png")):
    img = PILImage.open(f).resize((256, 256), PILImage.BICUBIC)
    img.save(f)
print("Done — all targets resized")

Done — all targets resized


In [18]:
p,s,l,f = compute_metrics(hd_recon, hd_target)
all_results["HistDiT+SAM"] = dict(psnr=p, ssim=s, lpips=l, fid=f, epoch=50, n=585)
print(f"  PSNR={p:.4f}  SSIM={s:.6f}  LPIPS={l:.4f}  FID={f:.2f}")

Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:00<00:00, 217MB/s]
100%|██████████| 37/37 [00:02<00:00, 15.17it/s]
/usr/local/lib/python3.12/dist-packages/pytorch_fid/fid_score.py:188: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)


  PSNR=11.9324  SSIM=0.000409  LPIPS=0.9403  FID=443.43


In [9]:
from PIL import Image as PILImage
from pathlib import Path

hd_target = Path("/content/results/analysis/histdit_target")
files = sorted(hd_target.glob("*.png"))
print(f"Total files: {len(files)}")
img = PILImage.open(files[0])
print(f"First image size: {img.size}")
img = PILImage.open(files[-1])
print(f"Last image size: {img.size}")

Total files: 585
First image size: (256, 256)
Last image size: (256, 256)


In [8]:
from PIL import Image as PILImage
from pathlib import Path

hd_target = Path("/content/results/analysis/histdit_target")
for f in sorted(hd_target.glob("*.png")):
    img = PILImage.open(f).resize((256, 256), PILImage.BICUBIC)
    img.save(f)
print("Done resizing targets")

Done resizing targets


In [5]:
from PIL import Image as PILImage
from pathlib import Path

hd_target = Path("/content/results/analysis/histdit_target")
for f in sorted(hd_target.glob("*.png")):
    img = PILImage.open(f)
    if img.size != (256, 256):
        img.resize((256, 256), PILImage.BICUBIC).save(f)
print("Done resizing targets")

Done resizing targets


In [3]:
import torch
DEVICE = torch.device("cuda")
ckpt = torch.load(
    "/content/drive/MyDrive/HistDiT_SAM_checkpoints_a100/epoch_0050.pt",
    map_location=DEVICE)
sam_enc.load_state_dict(ckpt["sam_enc"]); sam_enc.eval()
he_enc.load_state_dict(ckpt["he_enc"]);   he_enc.eval()
dit.load_state_dict(ckpt["dit"]);         dit.eval()

with torch.no_grad():
    dummy = torch.randn(1,3,512,512).to(DEVICE)
    dummy_t = torch.zeros(1,dtype=torch.long).to(DEVICE)
    sp,sg = sam_enc(dummy)
    print(f"SAM: sp={sp.shape} sg={sg.shape}")
    hf = he_enc(dummy)
    print(f"HE:  f1={hf[0].shape} f2={hf[1].shape} f3={hf[2].shape}")
    out = dit(torch.randn(1,3,512,512).to(DEVICE), dummy_t, sp, sg, hf)
    print(f"DiT output: {out.shape}")

SAM: sp=torch.Size([1, 256, 64, 64]) sg=torch.Size([1, 256])
HE:  f1=torch.Size([1, 64, 512, 512]) f2=torch.Size([1, 128, 256, 256]) f3=torch.Size([1, 256, 128, 128])
DiT output: torch.Size([1, 3, 512, 512])


In [22]:
ckpt = torch.load(
    "/content/drive/MyDrive/HistDiT_SAM_checkpoints_a100/epoch_0050.pt",
    map_location="cpu")
print(list(ckpt.keys()))

['epoch', 'opt_step', 'mode', 'sam_enc', 'he_enc', 'dit', 'opt', 'sched', 'scaler', 'val_psnr', 'val_ssim', 'config']


In [23]:
ckpt = torch.load(
    "/content/drive/MyDrive/HistDiT_SAM_checkpoints_a100/epoch_0050.pt",
    map_location="cpu")
print(ckpt["config"])

{'IMAGE_SIZE': 512, 'PATCH_SIZE': 16, 'DIT_DIM': 512, 'DIT_DEPTH': 12, 'DIT_HEADS': 8, 'NGF': 64, 'T_DIFF': 1000, 'MODE': 'a100', 'USE_REAL_SAM': False}
